# Ball Path Characteristics (Abhi)



This notebook builds pre-release 3D ball trajectories per shot using phase coordinates (`PreHitch -> Hitch -> PostHitch -> Release`) and phase times normalized by `timeAtStartofMovement`.



Key behaviors:

- Interpolating spline (no smoothing) fit per shot in 3D over normalized time

- Every trajectory is re-centered to start at the origin

- Interactive Plotly 3D visualization with filters for player and shot type

- Velocity-based coloring along each interpolated curve

In [1]:
import warnings

from pathlib import Path



import numpy as np

import pandas as pd

import plotly.graph_objects as go

from plotly.colors import sample_colorscale

from scipy.interpolate import PchipInterpolator



try:

    import ipywidgets as widgets

    from IPython.display import display

    HAS_WIDGETS = True

except Exception:

    HAS_WIDGETS = False



warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)

In [2]:
DATA_PATH = Path('..') / 'capstone2026v2.csv'

df = pd.read_csv(DATA_PATH)

print(f'Loaded rows: {len(df):,}; columns: {df.shape[1]}')


def pick_col(candidates, columns):

    return next((c for c in candidates if c in columns), None)



columns = set(df.columns)



PLAYER_CANDS = ['Name', 'Player', 'Player.Name']

SHOT_TYPE_CANDS = ['Shot.Type', 'ShotType', 'Shot Type']

TIME_START_CANDS = ['timeAtStartofMovement', 'timeAtStartOfMovement', 'TimeAtStartOfMovement']



player_col = pick_col(PLAYER_CANDS, columns)

shot_type_col = pick_col(SHOT_TYPE_CANDS, columns)

t_start_col = pick_col(TIME_START_CANDS, columns)



phase_order = ['PreHitch', 'Hitch', 'PostHitch', 'Release']

phase_xyz_cols = {

    'PreHitch': {'x': 'BallXPreHitch', 'y': 'BallYPreHitch', 'z': 'BallZPreHitch'},

    'Hitch': {'x': 'BallXHitch', 'y': 'BallYHitch', 'z': 'BallZHitch'},

    'PostHitch': {'x': 'BallXPostHitch', 'y': 'BallYPostHitch', 'z': 'BallZPostHitch'},

    'Release': {'x': 'BallXRelease', 'y': 'BallYRelease', 'z': 'BallZRelease'},

}

# Use strictly ordered event times to avoid duplicate-time spline instability.
phase_time_cols = {

    'PreHitch': 'timeAtStartofMovement',

    'Hitch': 'timeAtStartofHitch',

    'PostHitch': 'timeAtEndofHitch',

    'Release': 'timeAtRelease',

}

phase_vel_cols = {

    'PreHitch': 'BallVeloStart',

    'Hitch': 'BallVeloPreHitch',

    'PostHitch': 'BallVeloPostHitch',

    'Release': 'BallVeloRelease',

}

# Standardization strategy applied AFTER spline fitting.
# - 'none': no post-fit scaling.
# - 'player_axis_std': divide x/y/z by each player's axis-specific std.
POST_STANDARDIZATION = 'player_axis_std'



required_cols = []

required_cols.extend([v for p in phase_order for v in phase_xyz_cols[p].values()])

required_cols.extend([phase_time_cols[p] for p in phase_order])

required_cols.extend([phase_vel_cols[p] for p in phase_order])

required_cols.extend([player_col, shot_type_col, t_start_col])

required_cols = [c for c in required_cols if c is not None]



missing = [c for c in required_cols if c not in df.columns]

if missing:

    raise ValueError(f'Missing required columns: {missing}')



print('Using columns:')

print(f'  player: {player_col}')

print(f'  shot type: {shot_type_col}')

print(f'  movement start time: {t_start_col}')

print('  shooter proxy location: BallXPreHitch, BallYPreHitch, BallZPreHitch')

print('  basket is treated as (0, 0) in XY and each shot is rotated so pre-hitch->basket points +Y')

print(f'  post-fit standardization mode: {POST_STANDARDIZATION}')

Loaded rows: 25,868; columns: 264
Using columns:
  player: Name
  shot type: Shot.Type
  movement start time: timeAtStartofMovement
  shooter proxy location: BallXPreHitch, BallYPreHitch, BallZPreHitch
  basket is treated as (0, 0) in XY and each shot is rotated so pre-hitch->basket points +Y
  post-fit standardization mode: player_axis_std


In [3]:
# Build phase-level table per shot and normalize to movement start + pre-hitch-centered frame

shots = []

skip_counts = {
    'missing_values': 0,
    'time_not_monotonic': 0,
    'not_enough_unique_times': 0,
}

orientation_counts = {'rotated_to_basket_axis': 0, 'basket_at_origin': 0}



for idx, row in df.iterrows():

    x = np.array([row[phase_xyz_cols[p]['x']] for p in phase_order], dtype=float)

    y = np.array([row[phase_xyz_cols[p]['y']] for p in phase_order], dtype=float)

    z = np.array([row[phase_xyz_cols[p]['z']] for p in phase_order], dtype=float)

    t = np.array([row[phase_time_cols[p]] for p in phase_order], dtype=float)

    v = np.array([row[phase_vel_cols[p]] for p in phase_order], dtype=float)

    t0 = float(row[t_start_col])



    # Use pre-hitch ball position as the shooter proxy in ball-coordinate space.
    px = float(x[0])
    py = float(y[0])
    pz = float(z[0])



    if not np.isfinite(np.r_[x, y, z, t, v, t0, px, py, pz]).all():

        skip_counts['missing_values'] += 1

        continue



    t_norm = t - t0

    # Require strict temporal ordering to keep interpolation stable.
    if np.any(np.diff(t_norm) <= 0):

        skip_counts['time_not_monotonic'] += 1

        continue



    if np.unique(t_norm).size < 4:

        skip_counts['not_enough_unique_times'] += 1

        continue



    # Translate into pre-hitch-centered frame.
    x = x - px

    y = y - py

    z = z - pz

    # Rotate XY so the pre-hitch->basket vector aligns with +Y for all shots.
    # Basket is assumed at (0, 0) in this coordinate system.
    to_basket_x = -px
    to_basket_y = -py
    basket_dist = float(np.hypot(to_basket_x, to_basket_y))

    if basket_dist > 1e-8:
        theta = np.arctan2(to_basket_x, to_basket_y)
        c = np.cos(theta)
        s = np.sin(theta)
        x_rot = c * x - s * y
        y_rot = s * x + c * y
        x, y = x_rot, y_rot
        orientation_counts['rotated_to_basket_axis'] += 1
    else:
        orientation_counts['basket_at_origin'] += 1



    shots.append({

        'shot_id': int(idx),

        'player': row[player_col],

        'shot_type': row[shot_type_col],

        'phase': phase_order,

        't_phase': t_norm,

        't_phase_safe': t_norm,

        'x_phase': x,

        'y_phase': y,

        'z_phase': z,

        'v_phase': v,

    })



shots_df = pd.DataFrame(shots)

print(f'Valid shots: {len(shots_df):,}')

print('Skipped shots:', skip_counts)
print('Orientation ops:', orientation_counts)

Valid shots: 25,805
Skipped shots: {'missing_values': 0, 'time_not_monotonic': 63, 'not_enough_unique_times': 0}
Orientation ops: {'rotated_to_basket_axis': 25805, 'basket_at_origin': 0}


In [4]:
def interpolate_shot(row, n_samples=80, max_abs_coord=200.0):

    t = np.asarray(row['t_phase_safe'], dtype=float)

    x = np.asarray(row['x_phase'], dtype=float)

    y = np.asarray(row['y_phase'], dtype=float)

    z = np.asarray(row['z_phase'], dtype=float)

    v = np.asarray(row['v_phase'], dtype=float)



    t_dense = np.linspace(t.min(), t.max(), int(n_samples))

    # Shape-preserving interpolation avoids spline overshoot with sparse points.
    fx = PchipInterpolator(t, x)
    fy = PchipInterpolator(t, y)
    fz = PchipInterpolator(t, z)

    x_dense = fx(t_dense)
    y_dense = fy(t_dense)
    z_dense = fz(t_dense)

    # Safety fallback if any shot still produces unrealistic coordinates.
    if np.max(np.abs(np.r_[x_dense, y_dense, z_dense])) > max_abs_coord:
        x_dense = np.interp(t_dense, t, x)
        y_dense = np.interp(t_dense, t, y)
        z_dense = np.interp(t_dense, t, z)



    v_dense = np.interp(t_dense, t, v)



    return pd.DataFrame({

        'shot_id': int(row['shot_id']),

        'player': row['player'],

        'shot_type': row['shot_type'],

        't_norm': t_dense,

        'x': x_dense,

        'y': y_dense,

        'z': z_dense,

        'ball_velocity': v_dense,

    })



traj_parts = [interpolate_shot(row, n_samples=80) for _, row in shots_df.iterrows()]

traj_df = pd.concat(traj_parts, ignore_index=True) if traj_parts else pd.DataFrame()

# Standardize x/y/z within player AFTER interpolation.
if POST_STANDARDIZATION == 'player_axis_std' and not traj_df.empty:
    for axis in ['x', 'y', 'z']:
        player_scale = traj_df.groupby('player')[axis].transform('std')
        global_scale = float(traj_df[axis].std())
        player_scale = player_scale.fillna(global_scale)
        player_scale = player_scale.mask(player_scale.abs() < 1e-8, global_scale)
        if np.isfinite(global_scale) and global_scale > 1e-8:
            traj_df[axis] = traj_df[axis] / player_scale

    print('Applied post-fit player-wise axis standardization for x, y, z.')



print(f'Trajectory samples: {len(traj_df):,}')

if not traj_df.empty:

    starts = (traj_df.sort_values(['shot_id', 't_norm']).groupby('shot_id').first()[['x', 'y', 'z']])

    start_radius = np.sqrt((starts ** 2).sum(axis=1))

    coord_abs_max = float(traj_df[['x', 'y', 'z']].abs().to_numpy().max())

    print('Player-centered start distance stats (mean / median / max):',

          f"{start_radius.mean():.4f} / {start_radius.median():.4f} / {start_radius.max():.4f}")

    print(f'Max absolute coordinate after interpolation: {coord_abs_max:.4f}')

Applied post-fit player-wise axis standardization for x, y, z.
Trajectory samples: 2,064,400


Player-centered start distance stats (mean / median / max): 0.0000 / 0.0000 / 0.0000
Max absolute coordinate after interpolation: 18.4082


In [5]:
def velocity_to_color(values, cmin, cmax, colorscale='Turbo'):

    if cmax <= cmin:

        return ['rgb(128,128,128)'] * len(values)

    normed = np.clip((np.asarray(values) - cmin) / (cmax - cmin), 0.0, 1.0)

    return [sample_colorscale(colorscale, float(v))[0] for v in normed]



def make_trajectory_figure(data, max_shots=120, title_suffix=''):

    fig = go.Figure()

    if data.empty:

        fig.update_layout(title='No data after filters')

        return fig



    vel_min = float(data['ball_velocity'].min())

    vel_max = float(data['ball_velocity'].max())



    shot_ids = data['shot_id'].drop_duplicates().tolist()

    if len(shot_ids) > max_shots:

        shot_ids = shot_ids[:max_shots]

        data = data[data['shot_id'].isin(shot_ids)]



    for shot_id, g in data.groupby('shot_id', sort=False):

        g = g.sort_values('t_norm')

        x = g['x'].to_numpy()

        y = g['y'].to_numpy()

        z = g['z'].to_numpy()

        vel = g['ball_velocity'].to_numpy()



        seg_colors = velocity_to_color((vel[:-1] + vel[1:]) / 2.0, vel_min, vel_max, colorscale='Turbo')

        for i in range(len(x) - 1):

            fig.add_trace(go.Scatter3d(

                x=[x[i], x[i + 1]],

                y=[y[i], y[i + 1]],

                z=[z[i], z[i + 1]],

                mode='lines',

                line=dict(color=seg_colors[i], width=5),

                hoverinfo='skip',

                showlegend=False,

            ))



        fig.add_trace(go.Scatter3d(

            x=x,

            y=y,

            z=z,

            mode='markers',

            marker=dict(

                size=2,

                color=vel,

                colorscale='Turbo',

                cmin=vel_min,

                cmax=vel_max,

                showscale=False,

            ),

            customdata=np.stack([g['player'], g['shot_type'], g['t_norm']], axis=-1),

            hovertemplate=(

                'Shot ID: %{text}<br>'

                'Player: %{customdata[0]}<br>'

                'Shot Type: %{customdata[1]}<br>'

                't_norm: %{customdata[2]:.4f}s<br>'

                'Velocity: %{marker.color:.3f}<br>'

                'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'

            ),

            text=[shot_id] * len(g),

            showlegend=False,

        ))



    fig.add_trace(go.Scatter3d(

        x=[None], y=[None], z=[None],

        mode='markers',

        marker=dict(

            size=0.1,

            color=[vel_min],

            colorscale='Turbo',

            cmin=vel_min,

            cmax=vel_max,

            showscale=True,

            colorbar=dict(title='Ball Velocity')

        ),

        hoverinfo='skip',

        showlegend=False

    ))



    fig.update_layout(

        title=f'Pre-Release Ball Trajectories {title_suffix}'.strip(),

        scene=dict(

            xaxis_title='X (origin-shifted)',

            yaxis_title='Y (origin-shifted)',

            zaxis_title='Z (origin-shifted)',

            aspectmode='data',

        ),

        height=820,

        margin=dict(l=0, r=0, t=50, b=0),

    )

    return fig

In [6]:
all_players = sorted(traj_df['player'].dropna().astype(str).unique().tolist()) if not traj_df.empty else []

all_shot_types = sorted(traj_df['shot_type'].dropna().astype(str).unique().tolist()) if not traj_df.empty else []



print(f'Players: {len(all_players)} | Shot types: {len(all_shot_types)}')



def filter_trajectories(data, players=None, shot_types=None):

    if data.empty:

        return data

    out = data

    if players and 'All' not in players:

        out = out[out['player'].astype(str).isin(players)]

    if shot_types and 'All' not in shot_types:

        out = out[out['shot_type'].astype(str).isin(shot_types)]

    return out



if HAS_WIDGETS:

    player_options = ['All'] + all_players

    shot_options = ['All'] + all_shot_types



    player_select = widgets.SelectMultiple(

        options=player_options,

        value=('All',),

        description='Player',

        rows=min(12, max(6, len(player_options))),

        layout=widgets.Layout(width='360px')

    )

    shot_select = widgets.SelectMultiple(

        options=shot_options,

        value=('All',),

        description='Shot Type',

        rows=min(10, max(5, len(shot_options))),

        layout=widgets.Layout(width='360px')

    )

    max_shots_slider = widgets.IntSlider(

        value=120, min=20, max=500, step=20,

        description='Max Shots',

        continuous_update=False,

        layout=widgets.Layout(width='360px')

    )



    out = widgets.Output()



    def redraw(*_):

        out.clear_output(wait=True)

        f = filter_trajectories(

            traj_df,

            players=list(player_select.value),

            shot_types=list(shot_select.value),

        )

        suffix = f'| shots shown: {f["shot_id"].nunique()}' if not f.empty else ''

        fig = make_trajectory_figure(f, max_shots=max_shots_slider.value, title_suffix=suffix)

        with out:

            fig.show()



    player_select.observe(redraw, names='value')

    shot_select.observe(redraw, names='value')

    max_shots_slider.observe(redraw, names='value')



    controls = widgets.HBox([player_select, shot_select])

    display(widgets.VBox([controls, max_shots_slider, out]))

    redraw()

else:

    print('ipywidgets not available. Using a default all-data figure fallback...')

    fig = make_trajectory_figure(traj_df, max_shots=120)

    fig.show()

Players: 165 | Shot types: 2


## Catch-and-Shoot Clustering with Geomstats

This section clusters normalized catch-and-shoot trajectories using geometric shape alignment.

Pipeline:
1. Filter to catch-and-shoot shots.
2. Build one fixed-length 3D trajectory per shot.
3. Normalize velocity **within each shot** (to reduce distance-driven magnitude effects).
4. Align curves with Geomstats (SRV metric with rotation + reparametrization quotient).
5. Cluster aligned curves and inspect cluster-average trajectories.

### Notes on Velocity Normalization

Velocity is normalized **within each shot** as a z-score over that shot's sampled trajectory points before clustering features are formed.

This helps remove raw speed magnitude effects tied to distance, while still preserving *how* velocity changes through the shot path.

In [ ]:
# Off-the-dribble clustering (self-contained cell).
# Reuses traj_df from earlier pipeline and mirrors catch-and-shoot processing.

# Ensure geomstats compatibility in this environment.
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid

import geomstats.backend as gs
from geomstats.geometry.discrete_curves import DiscreteCurvesStartingAtOrigin, SRVMetric
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

if 'pick_optimal_kmeans_k' not in globals():
    def pick_optimal_kmeans_k(feature_matrix, k_candidates, random_state=42):
        n_samples = int(feature_matrix.shape[0])
        valid_k = sorted({int(k) for k in k_candidates if 2 <= int(k) < n_samples})
        if not valid_k:
            fallback_k = 2 if n_samples >= 3 else 1
            return fallback_k, pd.DataFrame()

        scores = []
        for k in valid_k:
            model = KMeans(n_clusters=k, random_state=random_state, n_init='auto')
            labels = model.fit_predict(feature_matrix)
            if len(np.unique(labels)) < 2:
                continue
            score = float(silhouette_score(feature_matrix, labels))
            scores.append({'k': int(k), 'silhouette': score})

        if not scores:
            return valid_k[0], pd.DataFrame()

        score_df = pd.DataFrame(scores).sort_values('k').reset_index(drop=True)
        best_row = score_df.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]
        return int(best_row['k']), score_df

OD_PATTERN = 'off the dribble'
K_SAMPLING_OD = 60
MAX_CLUSTER_SHOTS_OD = None  # Set an int (e.g., 1000) to cap for faster experimentation.
RANDOM_SEED_OD = 42
VELOCITY_WEIGHT_OD = 0.35
N_CLUSTERS_CANDIDATES_OD = [2, 3, 4, 5, 6]

shot_meta_od = (
    traj_df[['shot_id', 'player', 'shot_type']]
    .drop_duplicates()
    .assign(shot_type_l=lambda x: x['shot_type'].astype(str).str.lower())
)

is_od = shot_meta_od['shot_type_l'].str.contains(OD_PATTERN, na=False)
off_ids = shot_meta_od.loc[is_od, 'shot_id'].tolist()
off_df = traj_df[traj_df['shot_id'].isin(off_ids)].copy()

print(f'Off-the-dribble shots available: {len(off_ids):,}')

# Additional safeguard: normalize x/y/z within player right before path clustering.
for axis in ['x', 'y', 'z']:
    mu = off_df.groupby('player')[axis].transform('mean')
    sd = off_df.groupby('player')[axis].transform('std')
    global_sd = float(off_df[axis].std()) if not off_df.empty else 1.0
    if not np.isfinite(global_sd) or global_sd <= 1e-8:
        global_sd = 1.0
    sd = sd.fillna(global_sd)
    sd = sd.mask(sd.abs() < 1e-8, global_sd)
    off_df[axis] = (off_df[axis] - mu) / sd

curve_rows_od = []
for shot_id, g in off_df.groupby('shot_id', sort=False):
    g = g.sort_values('t_norm')

    t = g['t_norm'].to_numpy(dtype=float)
    x = g['x'].to_numpy(dtype=float)
    y = g['y'].to_numpy(dtype=float)
    z = g['z'].to_numpy(dtype=float)
    v = g['ball_velocity'].to_numpy(dtype=float)

    if len(t) < 5 or not np.isfinite(np.r_[t, x, y, z, v]).all():
        continue

    u = np.linspace(0.0, 1.0, len(t))
    u_dense = np.linspace(0.0, 1.0, K_SAMPLING_OD)

    x_dense = np.interp(u_dense, u, x)
    y_dense = np.interp(u_dense, u, y)
    z_dense = np.interp(u_dense, u, z)

    v_std = float(np.std(v))
    v_dense = np.interp(u_dense, u, v)
    if v_std > 1e-8:
        v_norm = (v_dense - float(np.mean(v))) / v_std
    else:
        v_norm = np.zeros_like(v_dense)

    meta = g[['player', 'shot_type']].iloc[0]
    curve_rows_od.append({
        'shot_id': int(shot_id),
        'player': meta['player'],
        'shot_type': meta['shot_type'],
        'curve_xyz': np.column_stack([x_dense, y_dense, z_dense]),
        'velocity_norm': v_norm,
    })

curves_df_od = pd.DataFrame(curve_rows_od)
if isinstance(MAX_CLUSTER_SHOTS_OD, int) and len(curves_df_od) > MAX_CLUSTER_SHOTS_OD:
    curves_df_od = curves_df_od.sample(MAX_CLUSTER_SHOTS_OD, random_state=RANDOM_SEED_OD).reset_index(drop=True)

if curves_df_od.empty:
    raise ValueError('No off-the-dribble trajectories available for clustering.')

curves_xyz_od = np.stack(curves_df_od['curve_xyz'].to_list(), axis=0)
vel_norm_od = np.stack(curves_df_od['velocity_norm'].to_list(), axis=0)

curves_r3_od = DiscreteCurvesStartingAtOrigin(
    ambient_dim=3,
    k_sampling_points=K_SAMPLING_OD,
    equip=False,
)

curves_proj_od = np.array(curves_r3_od.projection(gs.array(curves_xyz_od)))
curves_proj_od = np.array(curves_r3_od.normalize(gs.array(curves_proj_od)))

finite_mask_od = np.isfinite(curves_proj_od).all(axis=(1, 2))
if not finite_mask_od.all():
    curves_df_od = curves_df_od.loc[finite_mask_od].reset_index(drop=True)
    vel_norm_od = vel_norm_od[finite_mask_od]
    curves_proj_od = curves_proj_od[finite_mask_od]

curves_r3_od.equip_with_metric(SRVMetric)
curves_r3_od.equip_with_group_action('rotations')
curves_r3_od.equip_with_quotient()

template_od = gs.array(curves_proj_od[0])
aligned_curves_od = [np.array(template_od)]
align_failures_od = 0
for i in range(1, len(curves_proj_od)):
    p = gs.array(curves_proj_od[i])
    try:
        aligned = curves_r3_od.fiber_bundle.align(p, template_od)
        aligned_curves_od.append(np.array(aligned))
    except Exception:
        align_failures_od += 1
        aligned_curves_od.append(np.array(p))

aligned_curves_od = np.stack(aligned_curves_od, axis=0)

shape_features_od = aligned_curves_od.reshape(len(aligned_curves_od), -1)
feature_matrix_od = np.hstack([shape_features_od, VELOCITY_WEIGHT_OD * vel_norm_od])
feature_matrix_od = StandardScaler().fit_transform(feature_matrix_od)

N_CLUSTERS_OD, od_k_scores = pick_optimal_kmeans_k(
    feature_matrix_od,
    k_candidates=N_CLUSTERS_CANDIDATES_OD,
    random_state=RANDOM_SEED_OD,
)
kmeans_od = KMeans(n_clusters=N_CLUSTERS_OD, random_state=RANDOM_SEED_OD, n_init='auto')
labels_od = kmeans_od.fit_predict(feature_matrix_od)
curves_df_od['cluster'] = labels_od

print(f'Shots used for off-the-dribble clustering: {len(curves_df_od):,}')
print(f'Alignment fallbacks used: {align_failures_od}')
print(f'Selected optimal k (off-the-dribble): {N_CLUSTERS_OD}')
if not od_k_scores.empty:
    print('Silhouette scores by k:')
    display(od_k_scores.round(4))
print('Cluster counts:')
print(curves_df_od['cluster'].value_counts().sort_index())

# Plot centroid curves with velocity gradient (same style as trajectory plots).
cluster_means_od = []
for c in sorted(curves_df_od['cluster'].unique()):
    idx = curves_df_od.index[curves_df_od['cluster'] == c].to_numpy()
    mean_curve = aligned_curves_od[idx].mean(axis=0)
    mean_vel = vel_norm_od[idx].mean(axis=0)
    cluster_means_od.append((c, mean_curve, mean_vel, len(idx)))

vel_all_od = np.concatenate([mvel for _, _, mvel, _ in cluster_means_od])
vel_min_od = float(np.min(vel_all_od))
vel_max_od = float(np.max(vel_all_od))

fig_od = go.Figure()
for c, mean_curve, mean_vel, n_c in cluster_means_od:
    x = mean_curve[:, 0]
    y = mean_curve[:, 1]
    z = mean_curve[:, 2]

    seg_vel = (mean_vel[:-1] + mean_vel[1:]) / 2.0
    seg_colors = velocity_to_color(seg_vel, vel_min_od, vel_max_od, colorscale='Turbo')

    for i in range(len(x) - 1):
        fig_od.add_trace(go.Scatter3d(
            x=[x[i], x[i + 1]],
            y=[y[i], y[i + 1]],
            z=[z[i], z[i + 1]],
            mode='lines',
            line=dict(width=9, color=seg_colors[i]),
            hoverinfo='skip',
            showlegend=False,
        ))

    fig_od.add_trace(go.Scatter3d(
        x=x,
        y=y,
        z=z,
        mode='markers',
        marker=dict(
            size=3,
            color=mean_vel,
            colorscale='Turbo',
            cmin=vel_min_od,
            cmax=vel_max_od,
            showscale=False,
        ),
        name=f'Cluster {c} (n={n_c})',
        hovertemplate=(
            f'Cluster {c}<br>'
            'v_norm: %{marker.color:.3f}<br>'
            'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'
        ),
    ))

fig_od.add_trace(go.Scatter3d(
    x=[None], y=[None], z=[None],
    mode='markers',
    marker=dict(
        size=0.1,
        color=[vel_min_od],
        colorscale='Turbo',
        cmin=vel_min_od,
        cmax=vel_max_od,
        showscale=True,
        colorbar=dict(title='Centroid velocity (within-shot normalized)'),
    ),
    hoverinfo='skip',
    showlegend=False,
))

fig_od.update_layout(
    title='Off-the-Dribble: Mean Aligned Curve by Cluster (Velocity Gradient)',
    scene=dict(
        xaxis_title='X (post-fit standardized)',
        yaxis_title='Y (post-fit standardized)',
        zaxis_title='Z (post-fit standardized)',
        aspectmode='data',
    ),
    height=760,
    margin=dict(l=0, r=0, t=60, b=0),
)
fig_od.show()

display(curves_df_od[['shot_id', 'player', 'shot_type', 'cluster']].head(12))

## Off-the-Dribble Clustering with Geomstats

This repeats the same geometry-based clustering pipeline for off-the-dribble shots, including within-shot velocity normalization to reduce distance-driven speed effects.

In [ ]:
# Geomstats alignment + clustering on catch-and-shoot trajectories.
# This cell is resilient to notebook execution order.

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

if 'pick_optimal_kmeans_k' not in globals():
    from sklearn.metrics import silhouette_score

    def pick_optimal_kmeans_k(feature_matrix, k_candidates, random_state=42):
        n_samples = int(feature_matrix.shape[0])
        valid_k = sorted({int(k) for k in k_candidates if 2 <= int(k) < n_samples})

        if not valid_k:
            fallback_k = 2 if n_samples >= 3 else 1
            return fallback_k, pd.DataFrame()

        scores = []
        for k in valid_k:
            model = KMeans(n_clusters=k, random_state=random_state, n_init='auto')
            labels = model.fit_predict(feature_matrix)
            if len(np.unique(labels)) < 2:
                continue
            score = float(silhouette_score(feature_matrix, labels))
            scores.append({'k': int(k), 'silhouette': score})

        if not scores:
            return valid_k[0], pd.DataFrame()

        score_df = pd.DataFrame(scores).sort_values('k').reset_index(drop=True)
        best_row = score_df.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]
        return int(best_row['k']), score_df

if 'curves_df' not in globals() or curves_df.empty:
    CATCH_PATTERNS = ['catch and shoot', 'catch & shoot', 'catch-and-shoot']
    K_SAMPLING = 60
    MAX_CLUSTER_SHOTS = None
    RANDOM_SEED = 42
    VELOCITY_WEIGHT = 0.35

    shot_meta = (
        traj_df[['shot_id', 'player', 'shot_type']]
        .drop_duplicates()
        .assign(shot_type_l=lambda x: x['shot_type'].astype(str).str.lower())
    )

    is_cns = shot_meta['shot_type_l'].apply(lambda s: any(p in s for p in CATCH_PATTERNS))
    catch_ids = shot_meta.loc[is_cns, 'shot_id'].tolist()
    catch_df = traj_df[traj_df['shot_id'].isin(catch_ids)].copy()

    curve_rows = []
    for shot_id, g in catch_df.groupby('shot_id', sort=False):
        g = g.sort_values('t_norm')

        t = g['t_norm'].to_numpy(dtype=float)
        x = g['x'].to_numpy(dtype=float)
        y = g['y'].to_numpy(dtype=float)
        z = g['z'].to_numpy(dtype=float)
        v = g['ball_velocity'].to_numpy(dtype=float)

        if len(t) < 5 or not np.isfinite(np.r_[t, x, y, z, v]).all():
            continue

        u = np.linspace(0.0, 1.0, len(t))
        u_dense = np.linspace(0.0, 1.0, K_SAMPLING)

        x_dense = np.interp(u_dense, u, x)
        y_dense = np.interp(u_dense, u, y)
        z_dense = np.interp(u_dense, u, z)

        v_std = float(np.std(v))
        v_dense = np.interp(u_dense, u, v)
        if v_std > 1e-8:
            v_norm = (v_dense - float(np.mean(v))) / v_std
        else:
            v_norm = np.zeros_like(v_dense)

        meta = g[['player', 'shot_type']].iloc[0]
        curve_rows.append({
            'shot_id': int(shot_id),
            'player': meta['player'],
            'shot_type': meta['shot_type'],
            'curve_xyz': np.column_stack([x_dense, y_dense, z_dense]),
            'velocity_norm': v_norm,
        })

    curves_df = pd.DataFrame(curve_rows)
    if isinstance(MAX_CLUSTER_SHOTS, int) and len(curves_df) > MAX_CLUSTER_SHOTS:
        curves_df = curves_df.sample(MAX_CLUSTER_SHOTS, random_state=RANDOM_SEED).reset_index(drop=True)

if curves_df.empty:
    raise ValueError('No catch-and-shoot trajectories available for clustering.')

# Ensure geomstats backend objects exist even when this cell runs before dedicated import cells.
if 'gs' not in globals() or 'DiscreteCurvesStartingAtOrigin' not in globals() or 'SRVMetric' not in globals():
    if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
        np.trapz = np.trapezoid
    import geomstats.backend as gs
    from geomstats.geometry.discrete_curves import DiscreteCurvesStartingAtOrigin, SRVMetric

if 'K_SAMPLING' not in globals():
    K_SAMPLING = int(curves_df.iloc[0]['curve_xyz'].shape[0])
if 'VELOCITY_WEIGHT' not in globals():
    VELOCITY_WEIGHT = 0.35
if 'RANDOM_SEED' not in globals():
    RANDOM_SEED = 42

curves_xyz = np.stack(curves_df['curve_xyz'].to_list(), axis=0)
vel_norm = np.stack(curves_df['velocity_norm'].to_list(), axis=0)

curves_r3 = DiscreteCurvesStartingAtOrigin(
    ambient_dim=3,
    k_sampling_points=K_SAMPLING,
    equip=False,
)

curves_proj = np.array(curves_r3.projection(gs.array(curves_xyz)))
curves_proj = np.array(curves_r3.normalize(gs.array(curves_proj)))

# Keep only finite curves after projection/normalization.
finite_mask = np.isfinite(curves_proj).all(axis=(1, 2))
if not finite_mask.all():
    curves_df = curves_df.loc[finite_mask].reset_index(drop=True)
    vel_norm = vel_norm[finite_mask]
    curves_proj = curves_proj[finite_mask]

# Use SRV metric with rotation quotient for stable alignment on this dataset.
curves_r3.equip_with_metric(SRVMetric)
curves_r3.equip_with_group_action('rotations')
curves_r3.equip_with_quotient()

template = gs.array(curves_proj[0])
aligned_curves = [np.array(template)]
align_failures = 0
for i in range(1, len(curves_proj)):
    point = gs.array(curves_proj[i])
    try:
        aligned = curves_r3.fiber_bundle.align(point, template)
        aligned_curves.append(np.array(aligned))
    except Exception:
        align_failures += 1
        aligned_curves.append(np.array(point))

aligned_curves = np.stack(aligned_curves, axis=0)

shape_features = aligned_curves.reshape(len(aligned_curves), -1)
feature_matrix = np.hstack([shape_features, VELOCITY_WEIGHT * vel_norm])
feature_matrix = StandardScaler().fit_transform(feature_matrix)

N_CLUSTERS_CANDIDATES = [2, 3, 4, 5, 6]
N_CLUSTERS, cns_k_scores = pick_optimal_kmeans_k(
    feature_matrix,
    k_candidates=N_CLUSTERS_CANDIDATES,
    random_state=RANDOM_SEED,
)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init='auto')
cluster_labels = kmeans.fit_predict(feature_matrix)

curves_df['cluster'] = cluster_labels
print(f'Alignment fallbacks used: {align_failures}')
print(f'Selected optimal k (catch-and-shoot): {N_CLUSTERS}')
if not cns_k_scores.empty:
    print('Silhouette scores by k:')
    display(cns_k_scores.round(4))
print('Cluster counts:')
print(curves_df['cluster'].value_counts().sort_index())
print('\nPlayers per cluster:')
print(curves_df.groupby('cluster')['player'].nunique())

In [ ]:
# Visualize mean aligned trajectory per cluster (catch-and-shoot)
# using velocity gradient along each centroid curve.
required = ['curves_df', 'aligned_curves', 'vel_norm']
missing = [k for k in required if k not in globals()]

if missing:
    # This visualization cell is placed early in notebook order.
    # Re-run this cell after the clustering cell has created curves_df, aligned_curves, vel_norm.
    pass
else:
    cluster_means = []
    for c in sorted(curves_df['cluster'].unique()):
        idx = curves_df.index[curves_df['cluster'] == c].to_numpy()
        mean_curve = aligned_curves[idx].mean(axis=0)
        mean_vel = vel_norm[idx].mean(axis=0)
        cluster_means.append((c, mean_curve, mean_vel, len(idx)))

    vel_all = np.concatenate([mvel for _, _, mvel, _ in cluster_means])
    vel_min = float(np.min(vel_all))
    vel_max = float(np.max(vel_all))

    fig_cluster = go.Figure()

    for c, mean_curve, mean_vel, n_c in cluster_means:
        x = mean_curve[:, 0]
        y = mean_curve[:, 1]
        z = mean_curve[:, 2]

        # Segment-wise color from average velocity between adjacent points.
        seg_vel = (mean_vel[:-1] + mean_vel[1:]) / 2.0
        seg_colors = velocity_to_color(seg_vel, vel_min, vel_max, colorscale='Turbo')

        for i in range(len(x) - 1):
            fig_cluster.add_trace(go.Scatter3d(
                x=[x[i], x[i + 1]],
                y=[y[i], y[i + 1]],
                z=[z[i], z[i + 1]],
                mode='lines',
                line=dict(width=9, color=seg_colors[i]),
                hoverinfo='skip',
                showlegend=False,
            ))

        fig_cluster.add_trace(go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='markers',
            marker=dict(
                size=3,
                color=mean_vel,
                colorscale='Turbo',
                cmin=vel_min,
                cmax=vel_max,
                showscale=False,
            ),
            name=f'Cluster {c} (n={n_c})',
            hovertemplate=(
                f'Cluster {c}<br>'
                'v_norm: %{marker.color:.3f}<br>'
                'x,y,z: (%{x:.3f}, %{y:.3f}, %{z:.3f})<extra></extra>'
            ),
        ))

    # Shared colorbar for centroid velocity gradient.
    fig_cluster.add_trace(go.Scatter3d(
        x=[None], y=[None], z=[None],
        mode='markers',
        marker=dict(
            size=0.1,
            color=[vel_min],
            colorscale='Turbo',
            cmin=vel_min,
            cmax=vel_max,
            showscale=True,
            colorbar=dict(title='Centroid velocity (within-shot normalized)'),
        ),
        hoverinfo='skip',
        showlegend=False,
    ))

    fig_cluster.update_layout(
        title='Catch-and-Shoot: Mean Aligned Curve by Cluster (Velocity Gradient)',
        scene=dict(
            xaxis_title='X (post-fit standardized)',
            yaxis_title='Y (post-fit standardized)',
            zaxis_title='Z (post-fit standardized)',
            aspectmode='data',
        ),
        height=760,
        margin=dict(l=0, r=0, t=60, b=0),
    )
    fig_cluster.show()

    display(curves_df[['shot_id', 'player', 'shot_type', 'cluster']].head(12))

In [ ]:
# Prepare fixed-length catch-and-shoot trajectories and within-shot velocity normalization.
CATCH_PATTERNS = ['catch and shoot', 'catch & shoot', 'catch-and-shoot']
K_SAMPLING = 60
MAX_CLUSTER_SHOTS = None  # Set an int (e.g., 1000) to cap for faster experimentation.
RANDOM_SEED = 42
VELOCITY_WEIGHT = 0.35

shot_meta = (
    traj_df[['shot_id', 'player', 'shot_type']]
    .drop_duplicates()
    .assign(shot_type_l=lambda x: x['shot_type'].astype(str).str.lower())
)

is_cns = shot_meta['shot_type_l'].apply(
    lambda s: any(p in s for p in CATCH_PATTERNS)
)
catch_ids = shot_meta.loc[is_cns, 'shot_id'].tolist()

catch_df = traj_df[traj_df['shot_id'].isin(catch_ids)].copy()
print(f'Catch-and-shoot shots available: {len(catch_ids):,}')

# Additional safeguard: normalize x/y/z within player right before path clustering.
for axis in ['x', 'y', 'z']:
    mu = catch_df.groupby('player')[axis].transform('mean')
    sd = catch_df.groupby('player')[axis].transform('std')
    global_sd = float(catch_df[axis].std()) if not catch_df.empty else 1.0
    if not np.isfinite(global_sd) or global_sd <= 1e-8:
        global_sd = 1.0
    sd = sd.fillna(global_sd)
    sd = sd.mask(sd.abs() < 1e-8, global_sd)
    catch_df[axis] = (catch_df[axis] - mu) / sd

curve_rows = []
for shot_id, g in catch_df.groupby('shot_id', sort=False):
    g = g.sort_values('t_norm')

    t = g['t_norm'].to_numpy(dtype=float)
    x = g['x'].to_numpy(dtype=float)
    y = g['y'].to_numpy(dtype=float)
    z = g['z'].to_numpy(dtype=float)
    v = g['ball_velocity'].to_numpy(dtype=float)

    if len(t) < 5 or not np.isfinite(np.r_[t, x, y, z, v]).all():
        continue

    # Reparameterize by normalized progress to make every shot same sampling length.
    u = np.linspace(0.0, 1.0, len(t))
    u_dense = np.linspace(0.0, 1.0, K_SAMPLING)

    x_dense = np.interp(u_dense, u, x)
    y_dense = np.interp(u_dense, u, y)
    z_dense = np.interp(u_dense, u, z)

    # Normalize velocity *within shot* to reduce distance-driven speed magnitude effects.
    v_std = float(np.std(v))
    v_dense = np.interp(u_dense, u, v)
    if v_std > 1e-8:
        v_norm = (v_dense - float(np.mean(v))) / v_std
    else:
        v_norm = np.zeros_like(v_dense)

    meta = g[['player', 'shot_type']].iloc[0]
    curve_rows.append({
        'shot_id': int(shot_id),
        'player': meta['player'],
        'shot_type': meta['shot_type'],
        'curve_xyz': np.column_stack([x_dense, y_dense, z_dense]),
        'velocity_norm': v_norm,
    })

curves_df = pd.DataFrame(curve_rows)

if isinstance(MAX_CLUSTER_SHOTS, int) and len(curves_df) > MAX_CLUSTER_SHOTS:
    curves_df = curves_df.sample(MAX_CLUSTER_SHOTS, random_state=RANDOM_SEED).reset_index(drop=True)

print(f'Shots used for geomstats clustering: {len(curves_df):,}')

In [ ]:
# Compatibility shim: recent numpy versions expose trapezoid, while geomstats expects trapz.
if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid

import geomstats.backend as gs
from geomstats.geometry.discrete_curves import DiscreteCurvesStartingAtOrigin, SRVMetric
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler


def pick_optimal_kmeans_k(feature_matrix, k_candidates, random_state=42):
    """Pick k using silhouette score over valid candidate values."""
    n_samples = int(feature_matrix.shape[0])
    valid_k = sorted({int(k) for k in k_candidates if 2 <= int(k) < n_samples})

    if not valid_k:
        fallback_k = 2 if n_samples >= 3 else 1
        return fallback_k, pd.DataFrame()

    scores = []
    for k in valid_k:
        model = KMeans(n_clusters=k, random_state=random_state, n_init='auto')
        labels = model.fit_predict(feature_matrix)

        # Silhouette requires at least 2 non-empty clusters.
        if len(np.unique(labels)) < 2:
            continue

        score = float(silhouette_score(feature_matrix, labels))
        scores.append({'k': int(k), 'silhouette': score})

    if not scores:
        return valid_k[0], pd.DataFrame()

    score_df = pd.DataFrame(scores).sort_values('k').reset_index(drop=True)
    best_row = score_df.sort_values(['silhouette', 'k'], ascending=[False, True]).iloc[0]
    return int(best_row['k']), score_df


print('Geomstats + sklearn clustering imports loaded, including optimal-k helper.')

## Notes

- Phase-time mapping uses available timing columns to preserve the requested order `PreHitch -> Hitch -> PostHitch -> Release`.

- Interpolation uses a shape-preserving method and strict time monotonicity checks to avoid overshoot artifacts.

- Coordinates are translated so `PreHitch` is at the origin using `BallXPreHitch`, `BallYPreHitch`, and `BallZPreHitch`.

- Each shot is then rotated in XY so the pre-hitch-to-basket direction is always aligned with +Y.

- After spline fitting, x/y/z are standardized within player using axis-specific standard deviation (toggle with `POST_STANDARDIZATION`).

- Curves are rendered as short line segments colored by interpolated velocity to achieve along-path velocity coloring.

## Cluster Characteristics Analysis (Within Shot Type)

This section summarizes domain-specific trajectory characteristics **within each shot type** cluster using the aligned curves and within-shot normalized velocity profiles.

In [ ]:
from IPython.display import display


def ensure_cluster_labels(curves_df_local, aligned_curves_local, vel_norm_local, family_label):
    if 'cluster' in curves_df_local.columns:
        return curves_df_local

    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

    if 'pick_optimal_kmeans_k' in globals():
        n_clusters, _ = pick_optimal_kmeans_k(
            StandardScaler().fit_transform(
                np.hstack([aligned_curves_local.reshape(len(aligned_curves_local), -1), 0.35 * vel_norm_local])
            ),
            k_candidates=[2, 3, 4, 5, 6],
            random_state=42,
        )
    else:
        n_clusters = 2

    feat = np.hstack([aligned_curves_local.reshape(len(aligned_curves_local), -1), 0.35 * vel_norm_local])
    feat = StandardScaler().fit_transform(feat)

    labels = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto').fit_predict(feat)
    out = curves_df_local.copy()
    out['cluster'] = labels.astype(int)
    print(f'Rebuilt missing {family_label} cluster labels (k={n_clusters}) for profile analysis.')
    return out


def build_cluster_profile(curves_df_local, aligned_curves_local, vel_norm_local, label):
    rows = []
    n_pts = aligned_curves_local.shape[1]
    split = max(2, n_pts // 3)

    for i in range(len(curves_df_local)):
        c = aligned_curves_local[i]
        v = vel_norm_local[i]

        x = c[:, 0]
        y = c[:, 1]
        z = c[:, 2]

        seg = np.diff(c, axis=0)
        path_length = float(np.sum(np.linalg.norm(seg, axis=1)))

        rows.append({
            'cluster': int(curves_df_local.iloc[i]['cluster']),
            'shot_id': int(curves_df_local.iloc[i]['shot_id']),
            'player': curves_df_local.iloc[i]['player'],
            'forward_carry': float(y[-1] - y[0]),
            'lateral_span': float(np.max(np.abs(x))),
            'lateral_finish': float(x[-1] - x[0]),
            'vertical_lift': float(np.max(z) - z[0]),
            'release_height_norm': float(z[-1]),
            'path_length': path_length,
            'early_velocity_norm': float(np.mean(v[:split])),
            'late_velocity_norm': float(np.mean(v[-split:])),
            'velocity_ramp': float(np.mean(v[-split:]) - np.mean(v[:split])),
        })

    per_shot = pd.DataFrame(rows)

    profile = (
        per_shot.groupby('cluster')
        .agg(
            n_shots=('shot_id', 'count'),
            n_players=('player', 'nunique'),
            forward_carry=('forward_carry', 'mean'),
            lateral_span=('lateral_span', 'mean'),
            lateral_finish=('lateral_finish', 'mean'),
            vertical_lift=('vertical_lift', 'mean'),
            release_height_norm=('release_height_norm', 'mean'),
            path_length=('path_length', 'mean'),
            early_velocity_norm=('early_velocity_norm', 'mean'),
            late_velocity_norm=('late_velocity_norm', 'mean'),
            velocity_ramp=('velocity_ramp', 'mean'),
        )
        .reset_index()
        .sort_values('cluster')
    )

    print(f'\n=== {label}: Cluster Profile ===')
    display(profile.round(4))

    # Rank clusters for quick interpretation.
    rank_cols = ['forward_carry', 'lateral_span', 'vertical_lift', 'path_length', 'velocity_ramp']
    rank_table = profile[['cluster'] + rank_cols].copy()
    for col in rank_cols:
        rank_table[col + '_rank'] = rank_table[col].rank(ascending=False, method='dense').astype(int)

    print(f'=== {label}: Relative Ranking (1 = highest) ===')
    display(rank_table[['cluster'] + [c + '_rank' for c in rank_cols]])

    return per_shot, profile


# Catch-and-shoot profiles
if {'curves_df', 'aligned_curves', 'vel_norm'} <= set(globals().keys()):
    curves_df = ensure_cluster_labels(curves_df, aligned_curves, vel_norm, 'catch-and-shoot')
    cns_per_shot, cns_profile = build_cluster_profile(curves_df, aligned_curves, vel_norm, 'Catch-and-Shoot')
else:
    print('Catch-and-shoot clustering variables not found. Re-run the catch-and-shoot clustering cells first.')

# Off-the-dribble profiles
if {'curves_df_od', 'aligned_curves_od', 'vel_norm_od'} <= set(globals().keys()):
    curves_df_od = ensure_cluster_labels(curves_df_od, aligned_curves_od, vel_norm_od, 'off-the-dribble')
    od_per_shot, od_profile = build_cluster_profile(curves_df_od, aligned_curves_od, vel_norm_od, 'Off-the-Dribble')
else:
    print('Off-the-dribble clustering variables not found. Re-run the off-the-dribble clustering cell first.')

In [ ]:
def summarize_cluster_characteristics(profile_df, label):
    if profile_df is None or profile_df.empty:
        return [f'- {label}: no cluster profile available.']

    cols = ['forward_carry', 'lateral_span', 'vertical_lift', 'path_length', 'velocity_ramp']
    valid_cols = [c for c in cols if c in profile_df.columns]

    lines = [f'### {label}']

    for col in valid_cols:
        high_idx = profile_df[col].idxmax()
        low_idx = profile_df[col].idxmin()
        high_row = profile_df.loc[high_idx]
        low_row = profile_df.loc[low_idx]

        lines.append(
            f"- Highest {col.replace('_', ' ')}: Cluster {int(high_row['cluster'])} "
            f"({high_row[col]:.3f}) vs lowest Cluster {int(low_row['cluster'])} ({low_row[col]:.3f})."
        )

    return lines

analysis_lines = ['## Domain-Specific Cluster Interpretation']

if 'cns_profile' in globals():
    analysis_lines.extend(summarize_cluster_characteristics(cns_profile, 'Catch-and-Shoot'))

if 'od_profile' in globals():
    analysis_lines.extend(summarize_cluster_characteristics(od_profile, 'Off-the-Dribble'))

analysis_lines.extend([
    '### Practical Read of the Clusters',
    '- Clusters with higher forward carry and lower lateral span generally indicate more direct ball transfer into release.',
    '- Clusters with higher vertical lift and longer path length suggest more pronounced loading patterns before release.',
    '- Positive velocity ramp indicates acceleration into release; flatter or negative ramp can indicate earlier speed generation or deceleration near release.',
])

print('\n'.join(analysis_lines))

## Domain-Specific Cluster Characteristics (Polished Narrative)

Within shot type, the clustering separates distinct movement signatures in how the ball is loaded and delivered into release.

### Catch-and-Shoot
Cluster 1 represents the most direct forward transfer profile. It has the largest average forward carry and the longest average pre-release path, while maintaining the smallest lateral spread. This pattern is consistent with an efficient gather where the ball advances toward release with limited side-to-side deviation.

Cluster 0 captures a higher-lift, wider-sweep pattern. It shows the greatest vertical rise and the largest lateral span, indicating a more pronounced upward and side-to-side loading action before release.

Cluster 2 is the compact, late-acceleration profile. It has the shortest average path and lowest vertical lift, but the strongest velocity ramp into release. In practice, this looks like a tighter pre-release trajectory that speeds up more aggressively near the end of the motion.

### Off-the-Dribble
Cluster 1 is the drive-forward acceleration profile. It has the highest forward carry and strongest velocity ramp, but the lowest lift, shortest path, and smallest lateral span. This indicates a compact dribble-to-shot transfer that emphasizes late speed generation.

Cluster 2 is the high-lift lateral-creation profile. It has the largest lateral span and highest vertical lift, suggesting greater ball repositioning and upward loading before release.

Cluster 0 is the long-path control profile. It has the longest average path length with moderate forward carry and lift, consistent with a more extended but controlled pre-release route.

### Basketball Interpretation
These clusters likely reflect different functional shooting strategies rather than simple good/bad categories:

- Direct-transfer profiles emphasize line-to-target efficiency and reduced lateral movement.
- High-lift/lateral profiles may reflect added creation space, balance correction, or timing control before release.
- Compact late-ramp profiles emphasize acceleration near release and may support faster release timing.

Because clustering was run separately by shot type, these interpretations should be used for within-type comparisons (cluster vs cluster inside catch-and-shoot, and cluster vs cluster inside off-the-dribble), not for direct cross-type ranking.

In [ ]:
import re
import plotly.express as px
from IPython.display import display


def _build_velocity_features_from_traj(traj):
    rows = []
    for shot_id, g in traj.groupby('shot_id'):
        g = g.sort_values('t_norm')
        v = g['ball_velocity'].to_numpy(dtype=float)
        t = g['t_norm'].to_numpy(dtype=float)
        if len(v) < 6:
            continue

        n = len(v)
        one_third = max(2, n // 3)
        two_third = max(one_third + 1, (2 * n) // 3)

        early = v[:one_third]
        mid = v[one_third:two_third]
        late = v[two_third:]

        peak_idx = int(np.argmax(v))

        rows.append({
            'shot_id': int(shot_id),
            'v_start': float(v[0]),
            'v_end': float(v[-1]),
            'v_peak': float(np.max(v)),
            'v_mean': float(np.mean(v)),
            'v_auc': float(np.trapezoid(v, t)),
            'v_t_peak': float(t[peak_idx]),
            'v_early_mean': float(np.mean(early)),
            'v_mid_mean': float(np.mean(mid)) if len(mid) else np.nan,
            'v_late_mean': float(np.mean(late)) if len(late) else np.nan,
            'v_late_minus_early': float(np.mean(late) - np.mean(early)) if len(late) else np.nan,
            'v_end_minus_start': float(v[-1] - v[0]),
        })

    return pd.DataFrame(rows)


def _find_handedness_column(df_source):
    lookup = {str(c).strip().lower(): c for c in df_source.columns}
    for cand in ['hand', 'handedness', 'shooter_handedness', 'shooting_hand']:
        if cand in lookup:
            return lookup[cand]
    return None


def _dominance_sides(hand_value):
    if pd.isna(hand_value):
        return None, None

    h = str(hand_value).strip().lower()
    if h.startswith('r'):
        return 'Right', 'Left'
    if h.startswith('l'):
        return 'Left', 'Right'
    return None, None


def _discover_joint_phase_columns(columns):
    phase_re = re.compile(
        r'^(max)(?P<joint>Ankle|Knee|Hip|Torso|Shoulder|Elbow)'
        r'(?P<motion>[A-Za-z]+?)(?P<side>Left|Right)?(?P<avg>Avg)?'
        r'(?P<phase>PreHitch|Hitch|PostHitch|Release)$'
    )

    motion_map = {
        'Flexion': 'flexion',
        'Dorsiflexion': 'flexion',
        'Extension': 'extension',
        'Plantarflexion': 'extension',
    }

    records = []
    for c in columns:
        if c.startswith('timeTo'):
            continue
        m = phase_re.match(c)
        if not m:
            continue

        joint = m.group('joint')
        motion_raw = m.group('motion')
        phase = m.group('phase')
        side = m.group('side')

        motion_type = motion_map.get(motion_raw)
        if motion_type is None:
            continue

        # For bilateral joints, keep explicit Left/Right columns only (no Avg).
        if joint != 'Torso':
            if side not in ('Left', 'Right'):
                continue
        else:
            side = 'Center'

        records.append({
            'column': c,
            'joint': joint,
            'side': side,
            'motion_type': motion_type,
            'phase': phase,
        })

    return pd.DataFrame(records)


def _build_coordination_features(df_source, meta_cols, hand_col=None):
    col_map = _discover_joint_phase_columns(df_source.columns)
    if col_map.empty:
        return pd.DataFrame(), col_map

    hand_col = hand_col if hand_col in df_source.columns else _find_handedness_column(df_source)

    work = df_source.reset_index(drop=False).rename(columns={'index': 'shot_id'})
    keep = ['shot_id'] + [c for c in meta_cols if c in work.columns]
    if hand_col and hand_col not in keep:
        keep.append(hand_col)
    keep += col_map['column'].unique().tolist()
    work = work[keep].copy()

    phase_order_local = ['PreHitch', 'Hitch', 'PostHitch']
    feature_rows = []

    for _, row in work.iterrows():
        shot_id = int(row['shot_id'])
        base = {'shot_id': shot_id}
        for c in meta_cols:
            if c in row.index:
                base[c] = row[c]

        dominant_side, non_dominant_side = _dominance_sides(row[hand_col]) if hand_col and hand_col in row.index else (None, None)

        joint_level = []

        for joint in sorted(col_map['joint'].unique()):
            if joint == 'Torso':
                role_defs = [('torso', 'Center')]
            else:
                role_defs = [
                    ('dominant', dominant_side),
                    ('non_dominant', non_dominant_side),
                ]

            for role_label, target_side in role_defs:
                if target_side is None:
                    continue

                joint_cols = col_map[(col_map['joint'] == joint) & (col_map['side'] == target_side)]
                if joint_cols.empty:
                    continue

                flex_vals = []
                ext_vals = []
                for phase in phase_order_local:
                    flex_col = joint_cols[(joint_cols['motion_type'] == 'flexion') & (joint_cols['phase'] == phase)]['column']
                    ext_col = joint_cols[(joint_cols['motion_type'] == 'extension') & (joint_cols['phase'] == phase)]['column']

                    flex_vals.append(float(row[flex_col.iloc[0]]) if len(flex_col) else np.nan)
                    ext_vals.append(float(row[ext_col.iloc[0]]) if len(ext_col) else np.nan)

                flex = np.array(flex_vals, dtype=float)
                ext = np.array(ext_vals, dtype=float)

                if np.all(np.isnan(flex)) and np.all(np.isnan(ext)):
                    continue

                joint_key = joint if role_label == 'torso' else f'{joint}_{role_label}'
                joint_feat = {
                    'joint': joint_key,
                    'flex_range': float(np.nanmax(flex) - np.nanmin(flex)) if np.isfinite(flex).any() else np.nan,
                    'ext_range': float(np.nanmax(ext) - np.nanmin(ext)) if np.isfinite(ext).any() else np.nan,
                    'flex_post_minus_pre': float(flex[2] - flex[0]) if np.isfinite(flex[[0, 2]]).all() else np.nan,
                    'ext_post_minus_pre': float(ext[2] - ext[0]) if np.isfinite(ext[[0, 2]]).all() else np.nan,
                    'balance_shift_post_minus_pre': float((ext[2] - flex[2]) - (ext[0] - flex[0]))
                    if np.isfinite(np.r_[flex[[0, 2]], ext[[0, 2]]]).all() else np.nan,
                    'hitch_balance_shift': float((ext[1] - flex[1]) - (ext[0] - flex[0]))
                    if np.isfinite(np.r_[flex[[0, 1]], ext[[0, 1]]]).all() else np.nan,
                }
                joint_level.append(joint_feat)

                # Joint-specific features for interpretability.
                base[f'{joint_key}_balance_shift'] = joint_feat['balance_shift_post_minus_pre']
                base[f'{joint_key}_hitch_balance_shift'] = joint_feat['hitch_balance_shift']

        if not joint_level:
            continue

        joint_df = pd.DataFrame(joint_level)
        base['coord_joint_count'] = int(len(joint_df))
        base['coord_flex_range_mean'] = float(joint_df['flex_range'].mean())
        base['coord_ext_range_mean'] = float(joint_df['ext_range'].mean())
        base['coord_balance_shift_mean'] = float(joint_df['balance_shift_post_minus_pre'].mean())
        base['coord_hitch_balance_shift_mean'] = float(joint_df['hitch_balance_shift'].mean())

        feature_rows.append(base)

    return pd.DataFrame(feature_rows), col_map


if 'df' not in globals() or 'traj_df' not in globals():
    raise RuntimeError('Please run the data-loading and trajectory-building cells first (df and traj_df are required).')

hand_col = _find_handedness_column(df)
meta_candidates = [
    c for c in [
        player_col if 'player_col' in globals() else 'Name',
        shot_type_col if 'shot_type_col' in globals() else 'Shot.Type',
        hand_col,
    ]
    if isinstance(c, str)
]
coord_df, joint_col_map = _build_coordination_features(df, meta_candidates, hand_col=hand_col)
vel_df = _build_velocity_features_from_traj(traj_df)

if coord_df.empty:
    print('No usable joint phase flexion/extension columns were detected for coordination analysis.')
else:
    analysis_df = coord_df.merge(vel_df, on='shot_id', how='inner')
    print(f'Joint phase columns detected: {len(joint_col_map)}')
    if hand_col:
        print(f'Handedness column used: {hand_col}')
    print(f'Shots in coordination analysis: {len(analysis_df):,}')

    coord_cols = [
        c for c in analysis_df.columns
        if c.startswith('coord_') or c.endswith('_balance_shift') or c.endswith('_hitch_balance_shift')
    ]
    vel_cols = [
        'v_start', 'v_end', 'v_peak', 'v_mean', 'v_auc', 'v_t_peak',
        'v_early_mean', 'v_mid_mean', 'v_late_mean', 'v_late_minus_early', 'v_end_minus_start'
    ]
    vel_cols = [c for c in vel_cols if c in analysis_df.columns]

    corr = analysis_df[coord_cols + vel_cols].corr(method='spearman').loc[coord_cols, vel_cols]

    print('\nSpearman correlation: joint coordination vs velocity profile features')
    display(corr.round(3))

    pairs = (
        corr.stack()
        .reset_index()
        .rename(columns={'level_0': 'coord_feature', 'level_1': 'velocity_feature', 0: 'spearman_r'})
        .dropna()
    )
    pairs['abs_r'] = pairs['spearman_r'].abs()
    top_pairs = pairs.sort_values('abs_r', ascending=False).head(12)

    print('\nTop relationships by |Spearman r|')
    display(top_pairs[['coord_feature', 'velocity_feature', 'spearman_r']].round(3))

    if not top_pairs.empty:
        heatmap_data = top_pairs.pivot(index='coord_feature', columns='velocity_feature', values='spearman_r')
        fig_heat = px.imshow(
            heatmap_data,
            color_continuous_scale='RdBu_r',
            zmin=-1,
            zmax=1,
            aspect='auto',
            title='Strongest Joint Coordination vs Velocity-Profile Associations (Spearman)'
        )
        fig_heat.update_layout(height=620)
        fig_heat.show()

    # Save for downstream narrative/plots.
    joint_velocity_analysis_df = analysis_df
    joint_velocity_corr = corr
    joint_velocity_top_pairs = top_pairs

## Joint Phase Coordination vs. Velocity Profile Change

This section explores whether how joints coordinate flexion and extension across phases (PreHitch -> Hitch -> PostHitch) is associated with changes in the ball's velocity profile on the final normalized spline trajectories.

Analysis plan:

1. Detect phase-level joint flexion/extension columns (including ankle dorsiflexion/plantarflexion as flexion/extension analogs).
2. Build shot-level joint coordination features from phase-to-phase changes.
3. Compute velocity-profile change features from normalized spline trajectories in `traj_df`.
4. Quantify relationships using Spearman correlations and visualize the strongest effects.

## Interpreting Potential-to-Kinetic Transfer Into Ball Speed

A practical way to discuss your idea is to treat joint flexion as a loading state and extension as unloading/transfer into the ball-hand system.

At a conceptual level:

$$E_{k,ball} = \frac{1}{2} m v^2$$

Because ball mass is constant in this dataset, a usable proxy is:

$$E_{k,proxy} \propto v^2$$

So release transfer can be analyzed with:

$$\Delta E_{k,proxy} = v_{release}^2 - v_{start}^2$$

Biomechanical framing for your statement:

- As a joint approaches terminal extension, it has less remaining ROM to continue accelerating the chain.
- If extension is coordinated proximally-to-distally, kinetic transfer to the ball can still peak near release.
- If extension is completed too early, the chain may show reduced late-phase contribution to ball acceleration.

The next cell builds shot-level coordination proxies and tests how they relate to release kinetic proxy and late velocity gain.

In [ ]:
from scipy.stats import spearmanr


def _get_joint_phase_table(df_source):
    if 'joint_col_map' in globals() and isinstance(joint_col_map, pd.DataFrame) and not joint_col_map.empty:
        m = joint_col_map.copy()
    elif '_discover_joint_phase_columns' in globals():
        m = _discover_joint_phase_columns(df_source.columns)
    else:
        raise RuntimeError('Joint column discovery function is unavailable. Run the joint coordination section first.')

    return m[m['phase'].isin(['PreHitch', 'Hitch', 'PostHitch'])].copy()


def _joint_transfer_features(df_source):
    m = _get_joint_phase_table(df_source)
    hand_col = _find_handedness_column(df_source) if '_find_handedness_column' in globals() else None

    work = df_source.reset_index(drop=False).rename(columns={'index': 'shot_id'})

    v_start_col = phase_vel_cols.get('PreHitch', 'BallVeloStart') if 'phase_vel_cols' in globals() else 'BallVeloStart'
    v_rel_col = phase_vel_cols.get('Release', 'BallVeloRelease') if 'phase_vel_cols' in globals() else 'BallVeloRelease'

    needed = {'shot_id', v_start_col, v_rel_col}
    needed |= set(m['column'].tolist())
    if hand_col:
        needed.add(hand_col)

    missing = [c for c in needed if c not in work.columns]
    if missing:
        raise ValueError(f'Missing columns for transfer analysis: {missing[:10]}')

    rows = []
    eps = 1e-8

    for _, row in work.iterrows():
        out = {'shot_id': int(row['shot_id'])}

        v0 = float(row[v_start_col])
        vr = float(row[v_rel_col])
        if not np.isfinite([v0, vr]).all():
            continue

        out['v_start'] = v0
        out['v_release'] = vr
        out['ke_proxy_release'] = vr ** 2
        out['delta_ke_proxy'] = (vr ** 2) - (v0 ** 2)

        if hand_col and '_dominance_sides' in globals():
            dominant_side, non_dominant_side = _dominance_sides(row[hand_col])
        else:
            dominant_side, non_dominant_side = (None, None)

        joint_scores = []
        for joint_base in sorted(m['joint'].unique()):
            if joint_base == 'Torso':
                role_defs = [('torso', 'Center')]
            else:
                role_defs = [
                    ('dominant', dominant_side),
                    ('non_dominant', non_dominant_side),
                ]

            for role_label, target_side in role_defs:
                if target_side is None:
                    continue

                mj = m[(m['joint'] == joint_base) & (m['side'] == target_side)]
                if mj.empty:
                    continue

                def _pick(motion, phase):
                    hit = mj[(mj['motion_type'] == motion) & (mj['phase'] == phase)]['column']
                    return hit.iloc[0] if len(hit) else None

                fp = _pick('flexion', 'PreHitch')
                fh = _pick('flexion', 'Hitch')
                fph = _pick('flexion', 'PostHitch')
                ep = _pick('extension', 'PreHitch')
                eh = _pick('extension', 'Hitch')
                eph = _pick('extension', 'PostHitch')

                vals = [fp, fh, fph, ep, eh, eph]
                if any(v is None for v in vals):
                    continue

                f_pre, f_h, f_post, e_pre, e_h, e_post = [float(row[c]) for c in vals]
                if not np.isfinite([f_pre, f_h, f_post, e_pre, e_h, e_post]).all():
                    continue

                # Normalized progress toward extension and away from flexion across phases.
                flex_range = max(f_pre, f_h, f_post) - min(f_pre, f_h, f_post)
                ext_range = max(e_pre, e_h, e_post) - min(e_pre, e_h, e_post)

                ext_progress = (e_post - e_pre) / (ext_range + eps)
                flex_unload = (f_pre - f_post) / (flex_range + eps)
                transfer_score = 0.5 * (ext_progress + flex_unload)

                joint_name = joint_base if role_label == 'torso' else f'{joint_base}_{role_label}'
                out[f'{joint_name}_transfer_score'] = transfer_score
                out[f'{joint_name}_terminal_extension_level'] = e_post
                joint_scores.append(transfer_score)

        if not joint_scores:
            continue

        out['chain_transfer_mean'] = float(np.mean(joint_scores))
        out['chain_transfer_std'] = float(np.std(joint_scores))
        rows.append(out)

    return pd.DataFrame(rows)


transfer_df = _joint_transfer_features(df)

# Optionally merge trajectory-based late profile metrics if available.
if 'joint_velocity_analysis_df' in globals() and isinstance(joint_velocity_analysis_df, pd.DataFrame):
    extra_cols = [c for c in ['shot_id', 'v_late_minus_early', 'v_end_minus_start', 'v_t_peak'] if c in joint_velocity_analysis_df.columns]
    if len(extra_cols) > 1:
        transfer_df = transfer_df.merge(
            joint_velocity_analysis_df[extra_cols].drop_duplicates('shot_id'),
            on='shot_id',
            how='left',
        )

print(f'Shots with joint-transfer features: {len(transfer_df):,}')

transfer_features = [
    c for c in transfer_df.columns
    if c.endswith('_transfer_score') or c in ['chain_transfer_mean', 'chain_transfer_std']
]

targets = [c for c in ['ke_proxy_release', 'delta_ke_proxy', 'v_late_minus_early', 'v_end_minus_start'] if c in transfer_df.columns]

corr_rows = []
for f in transfer_features:
    for t in targets:
        sub = transfer_df[[f, t]].dropna()
        if len(sub) < 30:
            continue
        r, p = spearmanr(sub[f], sub[t])
        corr_rows.append({'feature': f, 'target': t, 'spearman_r': float(r), 'p_value': float(p), 'n': int(len(sub))})

transfer_corr = pd.DataFrame(corr_rows)
if transfer_corr.empty:
    print('Insufficient data to compute stable transfer correlations.')
else:
    transfer_corr['abs_r'] = transfer_corr['spearman_r'].abs()
    transfer_corr = transfer_corr.sort_values('abs_r', ascending=False)
    print('\nTop joint-transfer relationships to ball kinetic proxy outcomes')
    display(transfer_corr.head(15).round(4))

    top = transfer_corr.iloc[0]
    f = top['feature']
    t = top['target']
    plot_df = transfer_df[[f, t]].dropna().copy()

    slope, intercept = np.polyfit(plot_df[f], plot_df[t], 1)
    xs = np.linspace(plot_df[f].min(), plot_df[f].max(), 120)
    ys = slope * xs + intercept

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df[f], y=plot_df[t], mode='markers',
        marker=dict(size=5, opacity=0.35),
        name='Shots'
    ))
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode='lines',
        line=dict(width=3),
        name='Linear trend'
    ))
    fig.update_layout(
        title=f'Top Transfer Relationship: {f} vs {t}',
        xaxis_title=f,
        yaxis_title=t,
        height=560,
    )
    fig.show()

# Persist for later narrative use.
joint_transfer_df = transfer_df
joint_transfer_corr = transfer_corr if 'transfer_corr' in locals() else pd.DataFrame()

## Joint Transfer Scores by Individual and Cluster

This section breaks transfer scores down at two levels:

1. Per-shot and per-joint transfer score.
2. Per-individual (player) and per-joint transfer score.

Then, it compares those values across the previously identified clusters within each shot type (catch-and-shoot and off-the-dribble).

In [ ]:
import plotly.express as px


if 'joint_transfer_df' not in globals() or joint_transfer_df.empty:
    raise RuntimeError('Run the transfer analysis cell first to create joint_transfer_df.')

if 'curves_df' not in globals() or 'curves_df_od' not in globals():
    raise RuntimeError('Run both clustering sections first so curves_df and curves_df_od are available.')

# Build a unified shot->cluster mapping from prior clustering outputs.
cluster_map_cns = (
    curves_df[['shot_id', 'cluster', 'player', 'shot_type']]
    .drop_duplicates('shot_id')
    .assign(shot_family='catch_and_shoot')
)
cluster_map_od = (
    curves_df_od[['shot_id', 'cluster', 'player', 'shot_type']]
    .drop_duplicates('shot_id')
    .assign(shot_family='off_the_dribble')
)
cluster_map = pd.concat([cluster_map_cns, cluster_map_od], ignore_index=True)

transfer_cols = [
    c for c in joint_transfer_df.columns
    if c.endswith('_transfer_score') and c != 'chain_transfer_mean'
]
if not transfer_cols:
    raise ValueError('No per-joint transfer score columns found in joint_transfer_df.')

long_transfer = (
    joint_transfer_df[['shot_id'] + transfer_cols]
    .melt(id_vars='shot_id', var_name='joint_feature', value_name='transfer_score')
    .dropna(subset=['transfer_score'])
)
long_transfer['joint'] = long_transfer['joint_feature'].str.replace('_transfer_score', '', regex=False)

# Merge to clustered shots only.
joint_cluster_df = long_transfer.merge(cluster_map, on='shot_id', how='inner')

print(f'Shots with both transfer and cluster labels: {joint_cluster_df["shot_id"].nunique():,}')
print(f'Rows in per-joint long table: {len(joint_cluster_df):,}')

# 1) Cluster-level joint means (within shot family).
cluster_joint_summary = (
    joint_cluster_df.groupby(['shot_family', 'cluster', 'joint'])
    .agg(
        mean_transfer=('transfer_score', 'mean'),
        std_transfer=('transfer_score', 'std'),
        n_shots=('shot_id', 'nunique'),
        n_rows=('transfer_score', 'size'),
    )
    .reset_index()
    .sort_values(['shot_family', 'joint', 'cluster'])
)

print('\nCluster-level joint transfer summary')
display(cluster_joint_summary.round(4))

# 2) Individual-level joint means within each cluster.
player_joint_cluster_summary = (
    joint_cluster_df.groupby(['shot_family', 'cluster', 'player', 'joint'])
    .agg(
        mean_transfer=('transfer_score', 'mean'),
        n_shots=('shot_id', 'nunique'),
    )
    .reset_index()
)

print('\nPlayer-level joint transfer summary (first 40 rows)')
display(player_joint_cluster_summary.head(40).round(4))

# 3) Cluster separation effect size proxy per joint.
joint_effects = (
    cluster_joint_summary.groupby(['shot_family', 'joint'])['mean_transfer']
    .agg(cluster_range=lambda s: float(s.max() - s.min()),
         cluster_std=lambda s: float(s.std(ddof=0)))
    .reset_index()
    .sort_values(['shot_family', 'cluster_range'], ascending=[True, False])
)

print('\nJoints with largest between-cluster differences')
display(joint_effects.head(20).round(4))

# 4) Heatmaps by shot family.
for fam in sorted(cluster_joint_summary['shot_family'].unique()):
    fam_df = cluster_joint_summary[cluster_joint_summary['shot_family'] == fam]
    heat = fam_df.pivot(index='joint', columns='cluster', values='mean_transfer')
    fig = px.imshow(
        heat,
        color_continuous_scale='RdBu_r',
        aspect='auto',
        title=f'Average Joint Transfer Score by Cluster ({fam})',
        labels={'x': 'Cluster', 'y': 'Joint', 'color': 'Mean transfer score'}
    )
    fig.update_layout(height=520)
    fig.show()

# Keep outputs for downstream reporting.
joint_cluster_long_df = joint_cluster_df
joint_cluster_summary = cluster_joint_summary
player_joint_cluster_transfer = player_joint_cluster_summary
joint_cluster_effects = joint_effects

## Player Cluster-Space + Joint Energy Transfer Summary

This final table summarizes, for each player:

1. Where they lie in cluster space by shot type (catch-and-shoot, off-the-dribble): shots per cluster, cluster proportions, and dominant cluster.
2. Their mean joint transfer scores by shot type.

This is intended as a compact player profile for comparing movement strategy and transfer behavior across shot contexts.

In [ ]:
if 'joint_cluster_long_df' not in globals() or joint_cluster_long_df.empty:
    raise RuntimeError('Run the player/joint transfer-by-cluster cell first to create joint_cluster_long_df.')

# Base table of unique shot assignments with player + shot family + cluster.
player_cluster_shots = (
    joint_cluster_long_df[['shot_id', 'player', 'shot_family', 'cluster']]
    .drop_duplicates()
)

# Wide cluster count table: one column per shot-family x cluster.
cluster_counts = (
    player_cluster_shots
    .groupby(['player', 'shot_family', 'cluster'])['shot_id']
    .nunique()
    .rename('n_shots')
    .reset_index()
)

cluster_counts_wide = (
    cluster_counts
    .pivot_table(index='player', columns=['shot_family', 'cluster'], values='n_shots', fill_value=0)
)
cluster_counts_wide.columns = [f'{fam}_cluster_{int(cl)}_shots' for fam, cl in cluster_counts_wide.columns]
cluster_counts_wide = cluster_counts_wide.reset_index()

# Cluster proportions within each player x shot family.
cluster_props = cluster_counts.copy()
cluster_props['total_family_shots'] = cluster_props.groupby(['player', 'shot_family'])['n_shots'].transform('sum')
cluster_props['cluster_prop'] = np.where(
    cluster_props['total_family_shots'] > 0,
    cluster_props['n_shots'] / cluster_props['total_family_shots'],
    np.nan,
)

cluster_props_wide = (
    cluster_props
    .pivot_table(index='player', columns=['shot_family', 'cluster'], values='cluster_prop', fill_value=0.0)
)
cluster_props_wide.columns = [f'{fam}_cluster_{int(cl)}_prop' for fam, cl in cluster_props_wide.columns]
cluster_props_wide = cluster_props_wide.reset_index()

# Dominant cluster per player per shot family.
dominant_cluster = (
    cluster_props.sort_values(['player', 'shot_family', 'cluster_prop'], ascending=[True, True, False])
    .drop_duplicates(['player', 'shot_family'])[['player', 'shot_family', 'cluster']]
)
dominant_cluster['cluster'] = dominant_cluster['cluster'].astype(int)

dominant_cluster_wide = (
    dominant_cluster
    .pivot(index='player', columns='shot_family', values='cluster')
    .add_prefix('dominant_cluster_')
    .reset_index()
)

# Joint transfer means by player x shot family x joint.
player_joint_transfer = (
    joint_cluster_long_df
    .groupby(['player', 'shot_family', 'joint'])['transfer_score']
    .mean()
    .reset_index()
)

player_joint_transfer_wide = (
    player_joint_transfer
    .pivot_table(index='player', columns=['shot_family', 'joint'], values='transfer_score')
)
player_joint_transfer_wide.columns = [
    f'{fam}_{joint.lower()}_transfer_mean' for fam, joint in player_joint_transfer_wide.columns
]
player_joint_transfer_wide = player_joint_transfer_wide.reset_index()

# Total clustered shot counts by shot family.
family_totals = (
    player_cluster_shots
    .groupby(['player', 'shot_family'])['shot_id']
    .nunique()
    .rename('n_family_shots')
    .reset_index()
)
family_totals_wide = (
    family_totals
    .pivot(index='player', columns='shot_family', values='n_family_shots')
    .add_prefix('total_')
    .add_suffix('_shots')
    .reset_index()
)

# Final unified player summary table.
player_cluster_transfer_summary = (
    family_totals_wide
    .merge(dominant_cluster_wide, on='player', how='outer')
    .merge(cluster_counts_wide, on='player', how='outer')
    .merge(cluster_props_wide, on='player', how='outer')
    .merge(player_joint_transfer_wide, on='player', how='outer')
    .sort_values('player')
    .reset_index(drop=True)
)

print(f'Players in final summary: {len(player_cluster_transfer_summary):,}')
print(f'Columns in final summary: {player_cluster_transfer_summary.shape[1]}')

display(player_cluster_transfer_summary.round(4))

# Save for export/reporting.
final_player_cluster_transfer_summary = player_cluster_transfer_summary

In [ ]:
def build_cluster_movement_profiles(profile_df, shot_type_label):
    if profile_df is None or profile_df.empty:
        return pd.DataFrame()

    metrics = ['forward_carry', 'lateral_span', 'vertical_lift', 'path_length', 'velocity_ramp']
    avail = [m for m in metrics if m in profile_df.columns]
    if not avail:
        return pd.DataFrame()

    out = profile_df[['cluster'] + avail].copy().sort_values('cluster').reset_index(drop=True)

    # Z-score within shot type across clusters to identify dominant traits.
    for m in avail:
        s = out[m]
        std = float(s.std(ddof=0))
        if std > 1e-8:
            out[m + '_z'] = (s - float(s.mean())) / std
        else:
            out[m + '_z'] = 0.0

    def describe_cluster(r):
        tags = []

        if r.get('forward_carry_z', 0.0) >= 0.5:
            tags.append('direct-forward transfer')
        elif r.get('forward_carry_z', 0.0) <= -0.5:
            tags.append('reduced forward carry')

        if r.get('lateral_span_z', 0.0) >= 0.5:
            tags.append('lateral sweep')
        elif r.get('lateral_span_z', 0.0) <= -0.5:
            tags.append('compact lateral path')

        if r.get('vertical_lift_z', 0.0) >= 0.5:
            tags.append('high lift loading')
        elif r.get('vertical_lift_z', 0.0) <= -0.5:
            tags.append('low lift loading')

        if r.get('path_length_z', 0.0) >= 0.5:
            tags.append('long transfer path')
        elif r.get('path_length_z', 0.0) <= -0.5:
            tags.append('compact transfer path')

        if r.get('velocity_ramp_z', 0.0) >= 0.5:
            tags.append('late acceleration')
        elif r.get('velocity_ramp_z', 0.0) <= -0.5:
            tags.append('flatter acceleration')

        if not tags:
            tags = ['balanced/mixed profile']

        return '; '.join(tags)

    out['movement_profile'] = out.apply(describe_cluster, axis=1)
    out['shot_type_group'] = shot_type_label

    ordered_cols = (
        ['shot_type_group', 'cluster', 'movement_profile']
        + avail
        + [m + '_z' for m in avail]
    )

    return out[ordered_cols]


profile_tables = []
if 'cns_profile' in globals() and isinstance(cns_profile, pd.DataFrame):
    profile_tables.append(build_cluster_movement_profiles(cns_profile, 'catch_and_shoot'))
if 'od_profile' in globals() and isinstance(od_profile, pd.DataFrame):
    profile_tables.append(build_cluster_movement_profiles(od_profile, 'off_the_dribble'))

cluster_movement_profiles = pd.concat([p for p in profile_tables if not p.empty], ignore_index=True)

if cluster_movement_profiles.empty:
    print('No cluster profile tables found. Run the cluster profile cell first.')
else:
    print('Cluster movement profiles from cluster means')
    display(cluster_movement_profiles.round(4))

    print('\nReadable summary by shot type:')
    for shot_type_group, g in cluster_movement_profiles.groupby('shot_type_group'):
        print(f'\n{shot_type_group}:')
        for _, row in g.sort_values('cluster').iterrows():
            print(f"  Cluster {int(row['cluster'])}: {row['movement_profile']}")

## Cluster Movement Profiles From Cluster Means

This section defines movement profiles directly from cluster mean features.

Approach:

1. Use cluster-level means from each shot type (`cns_profile`, `od_profile`).
2. Standardize metrics within shot type (z-score across clusters).
3. Assign profile tags from dominant standardized traits.
4. Output one profile table per shot type for use in downstream player profiling.

In [ ]:
from itertools import combinations
from scipy.stats import kruskal, mannwhitneyu


if 'joint_cluster_long_df' not in globals() or joint_cluster_long_df.empty:
    raise RuntimeError('Run the joint transfer by cluster section first to create joint_cluster_long_df.')


def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([])
    order = np.argsort(p)
    ranked = p[order]
    q = ranked * m / (np.arange(1, m + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    out = np.empty_like(q)
    out[order] = q
    return out


def _epsilon_squared_kruskal(H, n, k):
    if n <= k:
        return np.nan
    return max(0.0, float((H - k + 1) / (n - k)))


def _rank_biserial_from_u(U, n1, n2):
    if n1 == 0 or n2 == 0:
        return np.nan
    return float(2.0 * U / (n1 * n2) - 1.0)


data = joint_cluster_long_df.dropna(subset=['transfer_score']).copy()

omnibus_rows = []
pairwise_rows = []

for (shot_family, joint), g in data.groupby(['shot_family', 'joint']):
    groups = []
    cluster_labels = []
    for c, gc in g.groupby('cluster'):
        vals = gc['transfer_score'].to_numpy(dtype=float)
        if len(vals) >= 20:
            groups.append(vals)
            cluster_labels.append(int(c))

    if len(groups) < 2:
        continue

    H, p = kruskal(*groups)
    n_total = int(sum(len(arr) for arr in groups))
    k = len(groups)
    eps2 = _epsilon_squared_kruskal(H, n_total, k)

    omnibus_rows.append({
        'shot_family': shot_family,
        'joint': joint,
        'n_total': n_total,
        'k_clusters': k,
        'H_stat': float(H),
        'p_value': float(p),
        'epsilon_sq': eps2,
    })

    cluster_to_vals = {cl: arr for cl, arr in zip(cluster_labels, groups)}
    for c1, c2 in combinations(sorted(cluster_to_vals.keys()), 2):
        a = cluster_to_vals[c1]
        b = cluster_to_vals[c2]
        U, p_u = mannwhitneyu(a, b, alternative='two-sided')
        rbc = _rank_biserial_from_u(U, len(a), len(b))
        pairwise_rows.append({
            'shot_family': shot_family,
            'joint': joint,
            'cluster_a': int(c1),
            'cluster_b': int(c2),
            'n_a': int(len(a)),
            'n_b': int(len(b)),
            'u_stat': float(U),
            'p_value': float(p_u),
            'rank_biserial': rbc,
            'mean_a': float(np.mean(a)),
            'mean_b': float(np.mean(b)),
            'mean_diff_a_minus_b': float(np.mean(a) - np.mean(b)),
        })

omnibus_df = pd.DataFrame(omnibus_rows)
pairwise_df = pd.DataFrame(pairwise_rows)

if omnibus_df.empty:
    print('No valid groups found for joint transfer cluster-difference testing.')
else:
    omnibus_df['q_value'] = _bh_fdr(omnibus_df['p_value'].to_numpy())
    omnibus_df['significant_fdr_05'] = omnibus_df['q_value'] < 0.05
    omnibus_df = omnibus_df.sort_values(['shot_family', 'q_value', 'p_value', 'joint']).reset_index(drop=True)

    print('Omnibus cluster-difference tests (Kruskal-Wallis)')
    display(omnibus_df.round(4))

    if not pairwise_df.empty:
        pairwise_df['q_value'] = _bh_fdr(pairwise_df['p_value'].to_numpy())
        pairwise_df['significant_fdr_05'] = pairwise_df['q_value'] < 0.05
        pairwise_df = pairwise_df.sort_values(['shot_family', 'joint', 'q_value', 'p_value']).reset_index(drop=True)

        sig_pw = pairwise_df[pairwise_df['significant_fdr_05']].copy()
        print('\nSignificant pairwise differences (Mann-Whitney, FDR < 0.05)')
        display(sig_pw.round(4))

        # Compact effect-size summary per joint.
        eff = (
            sig_pw.assign(abs_rbc=lambda d: d['rank_biserial'].abs())
            .groupby(['shot_family', 'joint'])['abs_rbc']
            .max()
            .reset_index()
            .sort_values(['shot_family', 'abs_rbc'], ascending=[True, False])
        )
        print('\nLargest pairwise effect size by joint (max |rank-biserial|)')
        display(eff.round(4))

# Persist results for downstream player-profile integration.
joint_cluster_omnibus_tests = omnibus_df if 'omnibus_df' in locals() else pd.DataFrame()
joint_cluster_pairwise_tests = pairwise_df if 'pairwise_df' in locals() else pd.DataFrame()

## Findings Summary: Joint Transfer Differences Across Clusters

### 1) Overall result
Joint transfer differs significantly by cluster in both shot families.

- For catch-and-shoot, all joints showed significant omnibus differences across clusters (Kruskal-Wallis, FDR < 0.05).
- For off-the-dribble, all joints also showed significant omnibus differences (Kruskal-Wallis, FDR < 0.05).

### 2) Strongest joints separating clusters
Using omnibus epsilon-squared and pairwise rank-biserial effect sizes:

- Catch-and-shoot: the strongest cluster separators were Knee, Hip, and Ankle transfer.
  - Largest pairwise effects were approximately: Knee 0.655, Hip 0.598, Ankle 0.563.
- Off-the-dribble: the strongest separator was Shoulder transfer, followed by Knee and Ankle.
  - Largest pairwise effects were approximately: Shoulder 0.436, Knee 0.334, Ankle 0.325.

### 3) Practical interpretation
- Catch-and-shoot cluster structure appears to be driven more by lower-body transfer organization (especially knee/hip/ankle).
- Off-the-dribble cluster structure appears to shift more influence toward upper-chain transfer differences (notably shoulder), while lower-body still contributes.
- Torso effects are present but generally smaller than the top separating joints.

### 4) How to use this in player profiling
For each shot type, treat the top separating joints as the primary "profile axes":

- Catch-and-shoot profile axes: Knee, Hip, Ankle.
- Off-the-dribble profile axes: Shoulder, Knee, Ankle.

These axes can be used to describe why a player maps to a given cluster beyond just trajectory shape.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


if 'joint_transfer_df' not in globals() or joint_transfer_df.empty:
    raise RuntimeError('Run the joint transfer feature cell first to create joint_transfer_df.')

if 'df' not in globals():
    raise RuntimeError('Source dataframe df is required.')

TRANSFER_K_CANDIDATES = [2, 3, 4, 5, 6]
TRANSFER_CLUSTER_RANDOM_SEED = 42

# Build shot metadata from source data using original index as shot_id.
meta_cols = []
if 'player_col' in globals() and player_col in df.columns:
    meta_cols.append(player_col)
if 'shot_type_col' in globals() and shot_type_col in df.columns:
    meta_cols.append(shot_type_col)

shot_meta_transfer = (
    df.reset_index(drop=False)
    .rename(columns={'index': 'shot_id'})[['shot_id'] + meta_cols]
    .rename(columns={player_col: 'player', shot_type_col: 'shot_type'})
)

transfer_feature_cols = [c for c in joint_transfer_df.columns if c.endswith('_transfer_score')]
if not transfer_feature_cols:
    raise ValueError('No joint transfer score columns found for clustering.')

transfer_shots = (
    joint_transfer_df[['shot_id'] + transfer_feature_cols]
    .merge(shot_meta_transfer, on='shot_id', how='left')
)
transfer_shots['shot_family'] = transfer_shots['shot_type'].astype(str).str.lower().map(
    lambda s: 'catch_and_shoot' if 'catch' in s else ('off_the_dribble' if 'off' in s else 'other')
)

clustered_parts = []
summary_tables = {}
player_tables = {}
overlap_tables = {}
transfer_k_selection_by_family = {}

for fam in ['catch_and_shoot', 'off_the_dribble']:
    fam_df = transfer_shots[transfer_shots['shot_family'] == fam].copy()
    fam_df = fam_df.dropna(subset=transfer_feature_cols)

    if len(fam_df) < 50:
        print(f'Skipping {fam}: not enough shots after filtering ({len(fam_df)}).')
        continue

    X = fam_df[transfer_feature_cols].to_numpy(dtype=float)
    Xs = StandardScaler().fit_transform(X)

    best_k, k_scores = pick_optimal_kmeans_k(
        Xs,
        k_candidates=TRANSFER_K_CANDIDATES,
        random_state=TRANSFER_CLUSTER_RANDOM_SEED,
    )
    transfer_k_selection_by_family[fam] = {
        'best_k': int(best_k),
        'scores': k_scores,
    }

    kmeans_t = KMeans(n_clusters=best_k, random_state=TRANSFER_CLUSTER_RANDOM_SEED, n_init='auto')
    fam_df['transfer_cluster'] = kmeans_t.fit_predict(Xs).astype(int)

    # Cluster counts
    counts = fam_df['transfer_cluster'].value_counts().sort_index().rename_axis('transfer_cluster').to_frame('n_shots')
    summary_tables[fam] = counts

    # Player composition table
    player_cluster = (
        fam_df.groupby(['player', 'transfer_cluster'])['shot_id']
        .nunique()
        .reset_index(name='n_shots')
    )
    player_cluster['player_total'] = player_cluster.groupby('player')['n_shots'].transform('sum')
    player_cluster['cluster_prop_within_player'] = np.where(
        player_cluster['player_total'] > 0,
        player_cluster['n_shots'] / player_cluster['player_total'],
        np.nan,
    )
    player_tables[fam] = player_cluster.sort_values(['player', 'transfer_cluster'])

    # Compare against trajectory-based clusters if available.
    if fam == 'catch_and_shoot' and 'curves_df' in globals():
        ref = curves_df[['shot_id', 'cluster']].drop_duplicates().rename(columns={'cluster': 'trajectory_cluster'})
    elif fam == 'off_the_dribble' and 'curves_df_od' in globals():
        ref = curves_df_od[['shot_id', 'cluster']].drop_duplicates().rename(columns={'cluster': 'trajectory_cluster'})
    else:
        ref = None

    if ref is not None and not ref.empty:
        overlap = (
            fam_df[['shot_id', 'transfer_cluster']]
            .merge(ref, on='shot_id', how='inner')
            .groupby(['transfer_cluster', 'trajectory_cluster'])['shot_id']
            .nunique()
            .unstack(fill_value=0)
            .sort_index()
        )
        overlap_tables[fam] = overlap

    clustered_parts.append(fam_df)

if not clustered_parts:
    raise RuntimeError('No transfer-based clustering results were produced.')

transfer_clustered_shots = pd.concat(clustered_parts, ignore_index=True)

print('Transfer-based clustering complete.')
print(f'Total clustered shots: {len(transfer_clustered_shots):,}')

for fam in ['catch_and_shoot', 'off_the_dribble']:
    if fam in transfer_k_selection_by_family:
        best_k = transfer_k_selection_by_family[fam]['best_k']
        print(f'\n{fam}: selected optimal k = {best_k}')
        scores = transfer_k_selection_by_family[fam]['scores']
        if isinstance(scores, pd.DataFrame) and not scores.empty:
            print('Silhouette scores by k:')
            display(scores.round(4))

    if fam in summary_tables:
        print(f'\n{fam}: transfer cluster counts')
        display(summary_tables[fam])

    if fam in player_tables:
        print(f'\n{fam}: player-by-transfer-cluster (first 40 rows)')
        display(player_tables[fam].head(40).round(4))

    if fam in overlap_tables:
        print(f'\n{fam}: overlap with trajectory-based clusters')
        display(overlap_tables[fam])

# Persist outputs for downstream narrative/use.
transfer_cluster_counts_by_family = summary_tables
transfer_cluster_player_tables = player_tables
transfer_cluster_overlap_with_trajectory = overlap_tables
transfer_cluster_k_selection = transfer_k_selection_by_family

## Shot Clustering Based on Joint Transfer Features

This section clusters shots using only joint transfer features (not trajectory-shape features), with clustering run separately by shot type:

1. Catch-and-shoot transfer clusters.
2. Off-the-dribble transfer clusters.

Outputs include:

- cluster counts,
- player-by-cluster composition,
- comparison versus existing trajectory-based clusters.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px


if 'joint_transfer_df' not in globals() or joint_transfer_df.empty:
    raise RuntimeError('Run the joint transfer feature cell first to create joint_transfer_df.')

if 'df' not in globals():
    raise RuntimeError('Source dataframe df is required for player/shot metadata.')

if 'player_col' not in globals() or 'shot_type_col' not in globals():
    raise RuntimeError('player_col and shot_type_col must be defined. Run the schema-mapping cell first.')

if 'transfer_clustered_shots' not in globals() or transfer_clustered_shots.empty:
    raise RuntimeError('Run the transfer-based clustering cell first to create transfer_clustered_shots.')

transfer_feature_cols = [c for c in joint_transfer_df.columns if c.endswith('_transfer_score')]
if len(transfer_feature_cols) < 2:
    raise ValueError('Need at least two transfer-score features for PCA.')

shot_meta = (
    df.reset_index(drop=False)
    .rename(columns={'index': 'shot_id'})[['shot_id', player_col, shot_type_col]]
    .rename(columns={player_col: 'player', shot_type_col: 'shot_type'})
)
shot_meta['shot_family'] = shot_meta['shot_type'].astype(str).str.lower().map(
    lambda s: 'catch_and_shoot' if 'catch' in s else ('off_the_dribble' if 'off' in s else 'other')
)

transfer_with_meta = (
    joint_transfer_df[['shot_id'] + transfer_feature_cols]
    .merge(shot_meta[['shot_id', 'player', 'shot_type', 'shot_family']], on='shot_id', how='left')
)

# Keep only the two modeled families for consistency with other sections.
transfer_with_meta = transfer_with_meta[transfer_with_meta['shot_family'].isin(['catch_and_shoot', 'off_the_dribble'])].copy()

if transfer_with_meta.empty:
    raise ValueError('No transfer shots available after shot-family filtering.')

# Dominant transfer cluster label per player x shot family for coloring.
transfer_cluster_mode_family = (
    transfer_clustered_shots.groupby(['player', 'shot_family', 'transfer_cluster'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
    .sort_values(['player', 'shot_family', 'n_shots', 'transfer_cluster'], ascending=[True, True, False, True])
    .drop_duplicates(['player', 'shot_family'])[['player', 'shot_family', 'transfer_cluster']]
    .rename(columns={'transfer_cluster': 'dominant_transfer_cluster'})
)

# Dominant transfer cluster label per player across families (family + cluster together).
transfer_cluster_mode_overall = (
    transfer_clustered_shots.groupby(['player', 'shot_family', 'transfer_cluster'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
    .assign(cluster_label=lambda d: d['shot_family'] + ':C' + d['transfer_cluster'].astype(str))
    .sort_values(['player', 'n_shots', 'cluster_label'], ascending=[True, False, True])
    .drop_duplicates('player')[['player', 'cluster_label']]
    .rename(columns={'cluster_label': 'dominant_transfer_cluster_label'})
)

# Player x shot-family centroids in transfer-feature space.
player_family_features = (
    transfer_with_meta.groupby(['player', 'shot_family'])[transfer_feature_cols]
    .mean()
    .reset_index()
)

player_family_counts = (
    transfer_with_meta.groupby(['player', 'shot_family'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
)

pf = player_family_features.merge(player_family_counts, on=['player', 'shot_family'], how='left')
pf = pf.merge(transfer_cluster_mode_family, on=['player', 'shot_family'], how='left')
pf = pf.dropna(subset=transfer_feature_cols).copy()
pf['display_player'] = pf['player'].astype(str).str.replace(r'(?i)^player\s+', '', regex=True)

if len(pf) < 3:
    raise ValueError('Not enough player-family points to run PCA.')

Xs_pf = StandardScaler().fit_transform(pf[transfer_feature_cols].to_numpy(dtype=float))
pca_pf = PCA(n_components=2, random_state=42)
pcs_pf = pca_pf.fit_transform(Xs_pf)

pf['pc1'] = pcs_pf[:, 0]
pf['pc2'] = pcs_pf[:, 1]

expl_pf = pca_pf.explained_variance_ratio_
print('Player-by-family PCA explained variance:')
print(f'  PC1: {expl_pf[0]:.3f}, PC2: {expl_pf[1]:.3f}, Total: {expl_pf[:2].sum():.3f}')

fig_pf = px.scatter(
    pf,
    x='pc1',
    y='pc2',
    color='dominant_transfer_cluster',
    symbol='shot_family',
    size='n_shots',
    hover_name='display_player',
    text='display_player',
    title='Players in Joint Transfer PCA Space (cluster-colored)',
    labels={
        'pc1': f'PC1 ({expl_pf[0] * 100:.1f}% var)',
        'pc2': f'PC2 ({expl_pf[1] * 100:.1f}% var)',
        'dominant_transfer_cluster': 'Dominant Transfer Cluster',
        'shot_family': 'Shot Family',
        'n_shots': 'Number of shots',
    },
)
fig_pf.update_traces(textposition='top center', marker=dict(opacity=0.84, line=dict(width=0.5, color='white')))
fig_pf.update_layout(height=700)
fig_pf.show()

# Overall player centroids across both families.
player_overall_features = (
    transfer_with_meta.groupby('player')[transfer_feature_cols]
    .mean()
    .reset_index()
)
player_overall_counts = (
    transfer_with_meta.groupby('player')['shot_id']
    .nunique()
    .reset_index(name='n_shots')
)

po = player_overall_features.merge(player_overall_counts, on='player', how='left')
po = po.merge(transfer_cluster_mode_overall, on='player', how='left')
po = po.dropna(subset=transfer_feature_cols).copy()
po['display_player'] = po['player'].astype(str).str.replace(r'(?i)^player\s+', '', regex=True)

if len(po) >= 3:
    Xs_po = StandardScaler().fit_transform(po[transfer_feature_cols].to_numpy(dtype=float))
    pca_po = PCA(n_components=2, random_state=42)
    pcs_po = pca_po.fit_transform(Xs_po)

    po['pc1'] = pcs_po[:, 0]
    po['pc2'] = pcs_po[:, 1]

    expl_po = pca_po.explained_variance_ratio_
    print('\nOverall-player PCA explained variance:')
    print(f'  PC1: {expl_po[0]:.3f}, PC2: {expl_po[1]:.3f}, Total: {expl_po[:2].sum():.3f}')

    fig_po = px.scatter(
        po,
        x='pc1',
        y='pc2',
        color='dominant_transfer_cluster_label',
        size='n_shots',
        hover_name='display_player',
        text='display_player',
        title='Players in Joint Transfer PCA Space (Overall, cluster-colored)',
        labels={
            'pc1': f'PC1 ({expl_po[0] * 100:.1f}% var)',
            'pc2': f'PC2 ({expl_po[1] * 100:.1f}% var)',
            'dominant_transfer_cluster_label': 'Dominant Transfer Cluster Label',
            'n_shots': 'Number of shots',
        },
    )
    fig_po.update_traces(textposition='top center', marker=dict(opacity=0.86, line=dict(width=0.5, color='white')))
    fig_po.update_layout(height=700)
    fig_po.show()
else:
    print('Skipping overall-player PCA plot: fewer than 3 players available.')

# Persist outputs for downstream use.
player_transfer_pca_by_family = pf
player_transfer_pca_overall = po

## Player Map in Joint Transfer PCA Space

This section projects player-level joint transfer signatures into 2D PCA space.

Views included:

1. Player-by-shot-family centroids in transfer space.
2. Overall player centroids across all shots.

Both plots are built from per-joint transfer score features.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px


required_objs = ['curves_df', 'aligned_curves', 'vel_norm', 'curves_df_od', 'aligned_curves_od', 'vel_norm_od']
missing_objs = [k for k in required_objs if k not in globals()]
if missing_objs:
    raise RuntimeError(f'Missing required trajectory objects for shot path PCA: {missing_objs}')

# Build per-shot path feature vectors using aligned shape and normalized velocity.
path_weight_velocity = 0.35

cns_meta = curves_df[['shot_id', 'player', 'cluster']].copy().rename(columns={'cluster': 'path_cluster'})
cns_meta['shot_family'] = 'catch_and_shoot'
cns_shape = aligned_curves.reshape(len(aligned_curves), -1)
cns_feat = np.hstack([cns_shape, path_weight_velocity * vel_norm])

od_meta = curves_df_od[['shot_id', 'player', 'cluster']].copy().rename(columns={'cluster': 'path_cluster'})
od_meta['shot_family'] = 'off_the_dribble'
od_shape = aligned_curves_od.reshape(len(aligned_curves_od), -1)
od_feat = np.hstack([od_shape, path_weight_velocity * vel_norm_od])

path_meta = pd.concat([cns_meta, od_meta], ignore_index=True)
path_feat = np.vstack([cns_feat, od_feat])

# Keep finite rows only.
finite_mask = np.isfinite(path_feat).all(axis=1)
path_meta = path_meta.loc[finite_mask].reset_index(drop=True)
path_feat = path_feat[finite_mask]

if len(path_meta) < 3:
    raise ValueError('Not enough finite shot-path rows for PCA.')

feat_cols = [f'f_{i}' for i in range(path_feat.shape[1])]
path_df = pd.concat([path_meta, pd.DataFrame(path_feat, columns=feat_cols)], axis=1)

# Dominant trajectory cluster per player x shot family for coloring.
path_cluster_mode_family = (
    path_df.groupby(['player', 'shot_family', 'path_cluster'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
    .sort_values(['player', 'shot_family', 'n_shots', 'path_cluster'], ascending=[True, True, False, True])
    .drop_duplicates(['player', 'shot_family'])[['player', 'shot_family', 'path_cluster']]
    .rename(columns={'path_cluster': 'dominant_path_cluster'})
)

# Dominant trajectory cluster label per player across families.
path_cluster_mode_overall = (
    path_df.groupby(['player', 'shot_family', 'path_cluster'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
    .assign(cluster_label=lambda d: d['shot_family'] + ':C' + d['path_cluster'].astype(str))
    .sort_values(['player', 'n_shots', 'cluster_label'], ascending=[True, False, True])
    .drop_duplicates('player')[['player', 'cluster_label']]
    .rename(columns={'cluster_label': 'dominant_path_cluster_label'})
)

# Player x shot-family centroids in shot-path feature space.
pf = (
    path_df.groupby(['player', 'shot_family'])[feat_cols]
    .mean()
    .reset_index()
)
pf_counts = (
    path_df.groupby(['player', 'shot_family'])['shot_id']
    .nunique()
    .reset_index(name='n_shots')
)
pf = pf.merge(pf_counts, on=['player', 'shot_family'], how='left')
pf = pf.merge(path_cluster_mode_family, on=['player', 'shot_family'], how='left')
pf['display_player'] = pf['player'].astype(str).str.replace(r'(?i)^player\s+', '', regex=True)

if len(pf) < 3:
    raise ValueError('Not enough player-family points to run shot-path PCA.')

Xs_pf = StandardScaler().fit_transform(pf[feat_cols].to_numpy(dtype=float))
pca_pf = PCA(n_components=2, random_state=42)
pcs_pf = pca_pf.fit_transform(Xs_pf)
pf['pc1'] = pcs_pf[:, 0]
pf['pc2'] = pcs_pf[:, 1]

expl_pf = pca_pf.explained_variance_ratio_
print('Shot-path player-by-family PCA explained variance:')
print(f'  PC1: {expl_pf[0]:.3f}, PC2: {expl_pf[1]:.3f}, Total: {expl_pf[:2].sum():.3f}')

fig_pf = px.scatter(
    pf,
    x='pc1',
    y='pc2',
    color='dominant_path_cluster',
    symbol='shot_family',
    size='n_shots',
    hover_name='display_player',
    text='display_player',
    title='Players in Shot Path PCA Space (cluster-colored)',
    labels={
        'pc1': f'PC1 ({expl_pf[0] * 100:.1f}% var)',
        'pc2': f'PC2 ({expl_pf[1] * 100:.1f}% var)',
        'dominant_path_cluster': 'Dominant Shot Path Cluster',
        'shot_family': 'Shot Family',
        'n_shots': 'Number of shots',
    },
)
fig_pf.update_traces(textposition='top center', marker=dict(opacity=0.84, line=dict(width=0.5, color='white')))
fig_pf.update_layout(height=700)
fig_pf.show()

# Overall player centroids.
po = (
    path_df.groupby('player')[feat_cols]
    .mean()
    .reset_index()
)
po_counts = (
    path_df.groupby('player')['shot_id']
    .nunique()
    .reset_index(name='n_shots')
)
po = po.merge(po_counts, on='player', how='left')
po = po.merge(path_cluster_mode_overall, on='player', how='left')
po['display_player'] = po['player'].astype(str).str.replace(r'(?i)^player\s+', '', regex=True)

if len(po) >= 3:
    Xs_po = StandardScaler().fit_transform(po[feat_cols].to_numpy(dtype=float))
    pca_po = PCA(n_components=2, random_state=42)
    pcs_po = pca_po.fit_transform(Xs_po)
    po['pc1'] = pcs_po[:, 0]
    po['pc2'] = pcs_po[:, 1]

    expl_po = pca_po.explained_variance_ratio_
    print('\nShot-path overall-player PCA explained variance:')
    print(f'  PC1: {expl_po[0]:.3f}, PC2: {expl_po[1]:.3f}, Total: {expl_po[:2].sum():.3f}')

    fig_po = px.scatter(
        po,
        x='pc1',
        y='pc2',
        color='dominant_path_cluster_label',
        size='n_shots',
        hover_name='display_player',
        text='display_player',
        title='Players in Shot Path PCA Space (Overall, cluster-colored)',
        labels={
            'pc1': f'PC1 ({expl_po[0] * 100:.1f}% var)',
            'pc2': f'PC2 ({expl_po[1] * 100:.1f}% var)',
            'dominant_path_cluster_label': 'Dominant Shot Path Cluster Label',
            'n_shots': 'Number of shots',
        },
    )
    fig_po.update_traces(textposition='top center', marker=dict(opacity=0.86, line=dict(width=0.5, color='white')))
    fig_po.update_layout(height=700)
    fig_po.show()
else:
    print('Skipping shot-path overall-player PCA plot: fewer than 3 players available.')

# Persist for downstream comparisons.
player_shotpath_pca_by_family = pf
player_shotpath_pca_overall = po

## Single-Shot Joint Ratio vs Ball Velocity (Interactive)

Use the controls below to select a player, shot type, and shot ID.

- Left panel: smoothed spline curves of dominant/non-dominant (plus torso) joint flexion-to-extension ratio over time.
- Right panel: smoothed ball-velocity curve (same interpolated trajectory velocity source used for ball-path coloring).

In [ ]:
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import PchipInterpolator


def _infer_hand_col(df_source):
    for c in ['hand', 'handedness', 'shooter_handedness', 'shooting_hand']:
        if c in df_source.columns:
            return c
    return None


def _dominant_side_from_hand(hand_value):
    if pd.isna(hand_value):
        return None
    h = str(hand_value).strip().lower()
    if h.startswith('r'):
        return 'Right'
    if h.startswith('l'):
        return 'Left'
    return None


def _joint_motion_tokens(joint_name):
    if joint_name == 'Ankle':
        return ('Dorsiflexion', 'Plantarflexion')
    return ('Flexion', 'Extension')


def _joint_series_for_side(row, joint_name, side_token=None, role_label=None):
    phases = ['PreHitch', 'Hitch', 'PostHitch']
    flex_tok, ext_tok = _joint_motion_tokens(joint_name)

    def _col(prefix, motion, phase):
        side = '' if side_token is None else side_token
        return f'{prefix}{joint_name}{motion}{side}{phase}'

    t_flex, y_flex = [], []
    t_ext, y_ext = [], []

    for ph in phases:
        c_flex = _col('max', flex_tok, ph)
        c_ext = _col('max', ext_tok, ph)
        c_tf = _col('timeToMax', flex_tok, ph)
        c_te = _col('timeToMax', ext_tok, ph)

        if c_flex in row.index and c_tf in row.index:
            vf, tf = row[c_flex], row[c_tf]
            if pd.notna(vf) and pd.notna(tf):
                y_flex.append(float(vf))
                t_flex.append(float(tf))

        if c_ext in row.index and c_te in row.index:
            ve, te = row[c_ext], row[c_te]
            if pd.notna(ve) and pd.notna(te):
                y_ext.append(float(ve))
                t_ext.append(float(te))

    if len(t_ext) < 2:
        return None

    t_ext = np.asarray(t_ext, dtype=float)
    y_ext = np.asarray(y_ext, dtype=float)

    # Ensure strictly increasing times for interpolation.
    t_ext, idx_e = np.unique(t_ext, return_index=True)
    y_ext = y_ext[idx_e]
    if len(t_ext) < 2:
        return None

    t_min = t_ext.min()
    t_max = t_ext.max()
    if not np.isfinite(t_min) or not np.isfinite(t_max) or t_max <= t_min:
        return None

    # Smoothed extension curve across joint event times.
    t_grid = np.linspace(t_min, t_max, 220)
    ext_spline = PchipInterpolator(t_ext, y_ext)
    ext_vals = ext_spline(t_grid)

    # Normalize by total ROM = extension peak + flexion peak for this joint-side shot.
    ext_peak = float(np.nanmax(np.abs(ext_vals))) if np.isfinite(ext_vals).any() else 0.0
    flex_peak = 0.0

    if len(t_flex) >= 2:
        t_flex = np.asarray(t_flex, dtype=float)
        y_flex = np.asarray(y_flex, dtype=float)
        t_flex, idx_f = np.unique(t_flex, return_index=True)
        y_flex = y_flex[idx_f]

        if len(t_flex) >= 2:
            flex_spline = PchipInterpolator(t_flex, y_flex)
            tf_min = max(t_flex.min(), t_ext.min())
            tf_max = min(t_flex.max(), t_ext.max())

            if np.isfinite(tf_min) and np.isfinite(tf_max) and tf_max > tf_min:
                t_rom = np.linspace(tf_min, tf_max, 220)
                ext_rom = ext_spline(t_rom)
                flex_rom = flex_spline(t_rom)
                ext_peak = float(np.nanmax(np.abs(ext_rom))) if np.isfinite(ext_rom).any() else ext_peak
                flex_peak = float(np.nanmax(np.abs(flex_rom))) if np.isfinite(flex_rom).any() else 0.0
            else:
                flex_vals = flex_spline(t_flex)
                flex_peak = float(np.nanmax(np.abs(flex_vals))) if np.isfinite(flex_vals).any() else 0.0

    rom_total = ext_peak + flex_peak
    if not np.isfinite(rom_total) or rom_total <= 1e-6:
        return None

    ext_norm = ext_vals / rom_total

    # Shift each joint curve so its first valid value starts at 0.
    finite_idx = np.flatnonzero(np.isfinite(ext_norm))
    if len(finite_idx) == 0:
        return None
    # ext_norm = ext_norm - ext_norm[finite_idx[0]]

    finite_ext = ext_norm[np.isfinite(ext_norm)]
    if len(finite_ext) < 3:
        return None

    series_name = f'{joint_name}_{role_label}' if role_label else joint_name
    return {
        'name': series_name,
        'time': t_grid,
        'ext': ext_norm,
    }


def _build_joint_extension_curves(row):
    hand_col_local = _infer_hand_col(df)
    dominant_side = _dominant_side_from_hand(row[hand_col_local]) if hand_col_local else None
    non_dom_side = 'Left' if dominant_side == 'Right' else ('Right' if dominant_side == 'Left' else None)

    curves = []

    # Bilateral joints mapped to dominant/non-dominant only.
    for j in ['Ankle', 'Knee', 'Hip', 'Shoulder', 'Elbow']:
        if dominant_side is not None:
            s_dom = _joint_series_for_side(row, j, side_token=dominant_side, role_label='dominant')
            if s_dom is not None:
                curves.append(s_dom)
        if non_dom_side is not None:
            s_nd = _joint_series_for_side(row, j, side_token=non_dom_side, role_label='non_dominant')
            if s_nd is not None:
                curves.append(s_nd)

    # Torso is centerline (not left/right), so plot unsided once.
    s_torso = _joint_series_for_side(row, 'Torso', side_token=None, role_label='torso')
    if s_torso is not None:
        curves.append(s_torso)

    return curves


def _shot_time_bounds_from_row(row):
    start_col = 'timeAtStartofMovement'
    end_col = 'timeAtRelease'

    if 'phase_time_cols' in globals() and isinstance(phase_time_cols, dict):
        start_col = phase_time_cols.get('PreHitch', start_col)
        end_col = phase_time_cols.get('Release', end_col)

    if start_col in row.index and end_col in row.index:
        t0 = row[start_col]
        t1 = row[end_col]
        if pd.notna(t0) and pd.notna(t1):
            t0 = float(t0)
            t1 = float(t1)
            if np.isfinite(t0) and np.isfinite(t1) and t1 > t0:
                return t0, t1

    return None


def _normalize_to_shot_time(t_values, row):
    t = np.asarray(t_values, dtype=float)
    bounds = _shot_time_bounds_from_row(row)
    if bounds is not None:
        t0, t1 = bounds
        return (t - t0) / (t1 - t0)

    finite = np.isfinite(t)
    if np.count_nonzero(finite) >= 2:
        tmin = float(np.nanmin(t[finite]))
        tmax = float(np.nanmax(t[finite]))
        if tmax > tmin:
            return (t - tmin) / (tmax - tmin)

    return t


def _raw_phase_times_for_row(row):
    phase_candidates = {
        'PreHitch': ['timeAtStartofMovement', 'timeAtStartOfMovement'],
        'Hitch': ['timeAtStartofHitch', 'timeAtStartOfHitch'],
        'PostHitch': ['timeAtEndofHitch', 'timeAtEndOfHitch'],
        'Release': ['timeAtRelease'],
    }

    if 'phase_time_cols' in globals() and isinstance(phase_time_cols, dict):
        for phase in ['PreHitch', 'Hitch', 'PostHitch', 'Release']:
            preferred = phase_time_cols.get(phase)
            if preferred:
                phase_candidates[phase] = [preferred] + [c for c in phase_candidates[phase] if c != preferred]

    raw = []
    for phase in ['PreHitch', 'Hitch', 'PostHitch', 'Release']:
        val = None
        for col in phase_candidates[phase]:
            if col in row.index and pd.notna(row[col]):
                v = float(row[col])
                if np.isfinite(v):
                    val = v
                    break
        if val is not None:
            raw.append((phase, val))

    return raw


def _phase_times_for_display(row, x_min, x_max):
    raw = _raw_phase_times_for_row(row)
    if len(raw) <= 1:
        return raw

    if x_min is None or x_max is None or not np.isfinite(x_min) or not np.isfinite(x_max) or x_max <= x_min:
        return raw

    t0 = next((v for p, v in raw if p == 'PreHitch'), None)
    t1 = next((v for p, v in raw if p == 'Release'), None)

    # Prefer progress-based mapping so all phase lines appear on the displayed x-axis.
    if t0 is not None and t1 is not None and np.isfinite(t0) and np.isfinite(t1) and t1 > t0:
        mapped = []
        for phase, tv in raw:
            prog = (tv - t0) / (t1 - t0)
            prog = float(np.clip(prog, 0.0, 1.0))
            x_disp = x_min + prog * (x_max - x_min)
            mapped.append((phase, x_disp))
        return mapped

    return raw


def _build_ball_velocity_component_curves(shot_id, t_eval_raw=None, row=None):
    if 'traj_df' not in globals() or traj_df is None or traj_df.empty:
        return []

    shot_traj = traj_df[traj_df['shot_id'] == int(shot_id)].copy()
    if shot_traj.empty or 't_norm' not in shot_traj.columns:
        return []

    shot_traj = shot_traj.sort_values('t_norm')

    # Use the precomputed/smoothed trajectory timeline from earlier notebook cells.
    t_norm_src = shot_traj['t_norm'].to_numpy(dtype=float)
    finite_t = np.isfinite(t_norm_src)
    t_norm_src = t_norm_src[finite_t]
    if len(t_norm_src) < 2:
        return []

    t_norm_src, idx = np.unique(t_norm_src, return_index=True)
    if len(t_norm_src) < 2:
        return []

    if t_eval_raw is None:
        if row is not None:
            bounds = _shot_time_bounds_from_row(row)
            if bounds is not None:
                t_eval_raw = np.linspace(bounds[0], bounds[1], 250)
            else:
                t_eval_raw = np.linspace(0.0, 1.0, 250)
        else:
            t_eval_raw = np.linspace(0.0, 1.0, 250)

    t_eval_raw = np.asarray(t_eval_raw, dtype=float)
    t_eval_raw = t_eval_raw[np.isfinite(t_eval_raw)]
    if len(t_eval_raw) == 0:
        return []
    t_eval_raw = np.unique(t_eval_raw)

    # Joint time samples can be either raw-time or already normalized.
    # Pick the mapping that gives best overlap with trajectory t_norm.
    t_eval_norm_raw = t_eval_raw.copy()
    t_eval_norm_mapped = _normalize_to_shot_time(t_eval_raw, row) if row is not None else t_eval_raw.copy()

    tt_min = float(np.nanmin(t_norm_src))
    tt_max = float(np.nanmax(t_norm_src))

    def _overlap_count(t_candidate):
        tc = np.asarray(t_candidate, dtype=float)
        tc = tc[np.isfinite(tc)]
        if len(tc) == 0:
            return 0
        return int(np.count_nonzero((tc >= tt_min) & (tc <= tt_max)))

    if _overlap_count(t_eval_norm_raw) >= _overlap_count(t_eval_norm_mapped):
        t_eval_norm = t_eval_norm_raw
    else:
        t_eval_norm = t_eval_norm_mapped

    def _interp_to_eval(y_src):
        y_src = np.asarray(y_src, dtype=float)
        y_src = y_src[finite_t]
        y_src = y_src[idx]
        good = np.isfinite(y_src)
        if np.count_nonzero(good) < 2:
            return None
        tt = t_norm_src[good]
        yy = y_src[good]
        out = np.full_like(t_eval_norm, np.nan, dtype=float)
        in_rng = (t_eval_norm >= tt.min()) & (t_eval_norm <= tt.max())
        if np.count_nonzero(in_rng):
            out[in_rng] = np.interp(t_eval_norm[in_rng], tt, yy)
        return out

    component_sets = [
        ('ball_velocity_x', 'ball_velocity_y', 'ball_velocity_z'),
        ('v_x', 'v_y', 'v_z'),
        ('vx', 'vy', 'vz'),
    ]

    curves = []
    labels = ['v_x', 'v_y', 'v_z']
    colors = ['#1f77b4', '#2ca02c', '#d62728']

    for cset in component_sets:
        if all(c in shot_traj.columns for c in cset):
            for c, lbl, color in zip(cset, labels, colors):
                v_eval = _interp_to_eval(shot_traj[c].to_numpy(dtype=float))
                if v_eval is None:
                    continue
                curves.append({'name': lbl, 'time': t_eval_raw, 'vel': v_eval, 'color': color})
            return curves

    # Fallback: derive components from already-smoothed x/y/z in traj_df, then interpolate.
    pos_cols = ['x', 'y', 'z']
    if not all(c in shot_traj.columns for c in pos_cols):
        return []

    for c, lbl, color in zip(pos_cols, labels, colors):
        pos = np.asarray(shot_traj[c].to_numpy(dtype=float))
        pos = pos[finite_t]
        pos = pos[idx]
        good = np.isfinite(pos)
        if np.count_nonzero(good) < 2:
            continue
        tt = t_norm_src[good]
        pp = pos[good]
        v_src = np.gradient(pp, tt)
        out = np.full_like(t_eval_norm, np.nan, dtype=float)
        in_rng = (t_eval_norm >= tt.min()) & (t_eval_norm <= tt.max())
        if np.count_nonzero(in_rng):
            out[in_rng] = np.interp(t_eval_norm[in_rng], tt, v_src)
        curves.append({'name': lbl, 'time': t_eval_raw, 'vel': out, 'color': color})

    return curves


def _make_single_shot_panel(shot_id):
    row = df.reset_index(drop=False).rename(columns={'index': 'shot_id'})
    row = row[row['shot_id'] == int(shot_id)]
    if row.empty:
        return None, f'Shot ID {shot_id} was not found.'

    row = row.iloc[0]

    ext_curves = _build_joint_extension_curves(row)

    # Evaluate velocity components at the same raw time samples used by the joint panel.
    if len(ext_curves):
        joint_raw_times = np.concatenate([c['time'] for c in ext_curves if len(c['time'])])
        joint_raw_times = joint_raw_times[np.isfinite(joint_raw_times)]
        joint_raw_times = np.unique(joint_raw_times)
    else:
        joint_raw_times = None

    vel_component_curves = _build_ball_velocity_component_curves(
        shot_id,
        t_eval_raw=joint_raw_times,
        row=row,
    )

    if len(ext_curves) == 0 and len(vel_component_curves) == 0:
        return None, 'No usable normalized extension or velocity data found for this shot.'

    # Compute x-range before phase marker placement.
    if len(ext_curves):
        all_joint_t = np.concatenate([c['time'] for c in ext_curves if len(c['time'])])
        finite_t = all_joint_t[np.isfinite(all_joint_t)]
        if len(finite_t):
            t_min = float(np.nanmin(finite_t))
            t_max = float(np.nanmax(finite_t))
        else:
            t_min, t_max = None, None
    else:
        t_min, t_max = None, None

    phase_times = _phase_times_for_display(row, x_min=t_min, x_max=t_max)

    fig = make_subplots(
        rows=1,
        cols=2,
        horizontal_spacing=0.08,
        subplot_titles=(
            'Joint Extension / Total ROM',
            'Ball Velocity Components',
        ),
    )

    for c in ext_curves:
        fig.add_trace(
            go.Scatter(
                x=c['time'],
                y=c['ext'],
                mode='lines',
                name=c['name'],
                line=dict(width=2),
            ),
            row=1,
            col=1,
        )

    for c in vel_component_curves:
        fig.add_trace(
            go.Scatter(
                x=c['time'],
                y=c['vel'],
                mode='lines',
                name=c['name'],
                line=dict(width=2.5, color=c['color']),
                showlegend=True,
            ),
            row=1,
            col=2,
        )

    # Demarcate shot phases on both panels with vertical guide lines.
    phase_styles = {
        'PreHitch': {'color': '#7f7f7f', 'dash': 'dot'},
        'Hitch': {'color': '#ff7f0e', 'dash': 'dash'},
        'PostHitch': {'color': '#2ca02c', 'dash': 'dash'},
        'Release': {'color': '#d62728', 'dash': 'solid'},
    }
    for phase, t_phase in phase_times:
        style = phase_styles.get(phase, {'color': '#666666', 'dash': 'dot'})
        for cidx in [1, 2]:
            fig.add_vline(
                x=t_phase,
                row=1,
                col=cidx,
                line_color=style['color'],
                line_dash=style['dash'],
                line_width=1.5,
                opacity=0.75,
            )
        fig.add_annotation(
            x=t_phase,
            y=-0.16,
            xref='x',
            yref='paper',
            text=phase,
            showarrow=False,
            font=dict(size=10, color=style['color']),
            xanchor='center',
            yanchor='top',
        )

    pcol = player_col if 'player_col' in globals() else 'Name'
    scol = shot_type_col if 'shot_type_col' in globals() else 'Shot.Type'
    hand_col_local = _infer_hand_col(df)
    hand_val = row[hand_col_local] if hand_col_local and hand_col_local in row.index else 'NA'

    fig.update_layout(
        title=f" Player: {row[pcol]} | Shot Type: {row[scol]} | Shot ID: {int(shot_id)} | Hand: {hand_val}",
        template='plotly_white',
        height=620,
        legend_title='Curves',
        hovermode='x unified',
        margin=dict(b=130),
    )

    # Align both panels to the same raw joint-time extent.
    if t_min is not None and t_max is not None and t_max > t_min:
        fig.update_xaxes(title_text='Raw Time', range=[t_min, t_max], row=1, col=1)
        fig.update_xaxes(title_text='Raw Time', range=[t_min, t_max], row=1, col=2)
    else:
        fig.update_xaxes(title_text='Raw Time', row=1, col=1)
        fig.update_xaxes(title_text='Raw Time', row=1, col=2)

    fig.update_yaxes(title_text='Extension / Total ROM (start at 0)', row=1, col=1)
    fig.update_yaxes(title_text='Velocity Component', row=1, col=2)

    return fig, None


def _single_shot_plotly_selector():
    if 'df' not in globals() or df is None or df.empty:
        raise RuntimeError('df is required. Run the data loading cells first.')
    if 'traj_df' not in globals() or traj_df is None or traj_df.empty:
        raise RuntimeError('traj_df is required. Run the trajectory section first.')

    pcol = player_col if 'player_col' in globals() else 'Name'
    scol = shot_type_col if 'shot_type_col' in globals() else 'Shot.Type'

    meta = (
        df.reset_index(drop=False)
        .rename(columns={'index': 'shot_id', pcol: 'player', scol: 'shot_type'})[['shot_id', 'player', 'shot_type']]
        .dropna(subset=['player', 'shot_type'])
        .copy()
    )

    meta['player'] = meta['player'].astype(str)
    meta['shot_type'] = meta['shot_type'].astype(str)
    meta['shot_id'] = meta['shot_id'].astype(int)
    meta = meta.sort_values(['player', 'shot_type', 'shot_id']).reset_index(drop=True)

    if meta.empty:
        raise RuntimeError('No shot metadata available for selector.')

    shot_ids = meta['shot_id'].tolist()

    frames = []
    valid_shot_ids = []
    for sid in shot_ids:
        shot_fig, err = _make_single_shot_panel(int(sid))
        if err is not None or shot_fig is None:
            continue

        frame_layout = go.Layout(
            title=shot_fig.layout.title,
            shapes=shot_fig.layout.shapes,
            annotations=shot_fig.layout.annotations,
            xaxis=shot_fig.layout.xaxis,
            xaxis2=shot_fig.layout.xaxis2,
            yaxis=shot_fig.layout.yaxis,
            yaxis2=shot_fig.layout.yaxis2,
        )
        frames.append(go.Frame(name=str(int(sid)), data=shot_fig.data, layout=frame_layout))
        valid_shot_ids.append(int(sid))

    if len(frames) == 0:
        raise RuntimeError('No plottable shots found for selector.')

    initial_sid = valid_shot_ids[0]
    base_fig, _ = _make_single_shot_panel(initial_sid)
    selector_fig = go.Figure(data=base_fig.data, layout=base_fig.layout, frames=frames)

    valid_meta = meta[meta['shot_id'].isin(valid_shot_ids)].copy()

    shot_buttons = []
    for _, r in valid_meta.iterrows():
        sid = int(r['shot_id'])
        label = f"{r['player']} | {r['shot_type']} | {sid}"
        shot_buttons.append(
            dict(
                label=label,
                method='animate',
                args=[[str(sid)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True}, 'transition': {'duration': 0}}],
            )
        )

    player_buttons = []
    for player in sorted(valid_meta['player'].unique().tolist()):
        sid = int(valid_meta[valid_meta['player'] == player]['shot_id'].iloc[0])
        player_buttons.append(
            dict(
                label=player,
                method='animate',
                args=[[str(sid)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True}, 'transition': {'duration': 0}}],
            )
        )

    shot_type_buttons = []
    for stype in sorted(valid_meta['shot_type'].unique().tolist()):
        sid = int(valid_meta[valid_meta['shot_type'] == stype]['shot_id'].iloc[0])
        shot_type_buttons.append(
            dict(
                label=stype,
                method='animate',
                args=[[str(sid)], {'mode': 'immediate', 'frame': {'duration': 0, 'redraw': True}, 'transition': {'duration': 0}}],
            )
        )

    selector_fig.update_layout(
        updatemenus=[
            dict(type='dropdown', direction='down', x=0.00, y=1.30, xanchor='left', yanchor='top', showactive=True, buttons=player_buttons, pad={'r': 8, 't': 2}),
            dict(type='dropdown', direction='down', x=0.33, y=1.30, xanchor='left', yanchor='top', showactive=True, buttons=shot_type_buttons, pad={'r': 8, 't': 2}),
            dict(type='dropdown', direction='down', x=0.66, y=1.30, xanchor='left', yanchor='top', showactive=True, buttons=shot_buttons, pad={'r': 8, 't': 2}),
        ],
        annotations=[
            dict(text='Player', x=0.00, y=1.35, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=11)),
            dict(text='Shot Type', x=0.33, y=1.35, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=11)),
            dict(text='Shot ID (Player | Type | ID)', x=0.66, y=1.35, xref='paper', yref='paper', showarrow=False, xanchor='left', font=dict(size=11)),
        ] + list(selector_fig.layout.annotations),
    )

    selector_fig.show()


# Disabled: precomputing all shots is too slow for interactive use.
# _single_shot_plotly_selector()

In [ ]:
# Selector-only launcher for the single-shot diagnostic panel.
# Run this cell to open sequential dropdowns: Player -> Shot Type -> Shot ID.
if 'HAS_WIDGETS' in globals() and HAS_WIDGETS:
    _single_shot_plotly_selector()
else:
    print('Interactive dropdowns require ipywidgets in this notebook kernel.')
    print('Current state: HAS_WIDGETS =', globals().get('HAS_WIDGETS', None))
    print('If needed, install ipywidgets and re-run the import/setup cell.')

In [ ]:
# Dash version: dynamic on-demand rendering (no precomputed all-shot frames)
from dash import Dash, dcc, html, Input, Output
import socket
import re

if 'df' not in globals() or df is None or df.empty:
    raise RuntimeError('df is required. Run the data loading cells first.')
if 'traj_df' not in globals() or traj_df is None or traj_df.empty:
    raise RuntimeError('traj_df is required. Run the trajectory section first.')

pcol = player_col if 'player_col' in globals() else 'Name'
scol = shot_type_col if 'shot_type_col' in globals() else 'Shot.Type'

meta = (
    df.reset_index(drop=False)
    .rename(columns={'index': 'shot_id', pcol: 'player', scol: 'shot_type'})[['shot_id', 'player', 'shot_type']]
    .dropna(subset=['player', 'shot_type'])
    .copy()
)
meta['player'] = meta['player'].astype(str)
meta['shot_type'] = meta['shot_type'].astype(str)
meta['shot_id'] = meta['shot_id'].astype(int)
meta = meta.sort_values(['player', 'shot_type', 'shot_id']).reset_index(drop=True)

players = sorted(meta['player'].unique().tolist())
first_player = players[0]
first_types = sorted(meta[meta['player'] == first_player]['shot_type'].unique().tolist())
first_type = first_types[0]
first_ids = meta[(meta['player'] == first_player) & (meta['shot_type'] == first_type)]['shot_id'].astype(int).tolist()
first_id = int(first_ids[0])


def _player_label(p):
    m = re.match(r'^Player\s+(\d+)$', str(p))
    if m:
        return m.group(1)
    return str(p)


app = Dash('single_shot_selector_dash')

app.layout = html.Div([
    html.H4('Single-Shot Selector (Dash, dynamic)'),
    html.Div([
        html.Div([
            html.Label('Player'),
            dcc.Dropdown(
                id='player-dd',
                options=[{'label': _player_label(p), 'value': p} for p in players],
                value=first_player,
                clearable=False,
            ),
        ], style={'width': '32%', 'display': 'inline-block', 'paddingRight': '10px'}),
        html.Div([
            html.Label('Shot Type'),
            dcc.Dropdown(id='shot-type-dd', clearable=False),
        ], style={'width': '32%', 'display': 'inline-block', 'paddingRight': '10px'}),
        html.Div([
            html.Label('Shot ID'),
            dcc.Dropdown(id='shot-id-dd', clearable=False),
        ], style={'width': '32%', 'display': 'inline-block'}),
    ], style={'marginBottom': '12px'}),
    html.Div(id='selection-readout', style={'marginBottom': '8px', 'fontSize': '13px'}),
    dcc.Graph(id='shot-graph'),
])


@app.callback(
    Output('shot-type-dd', 'options'),
    Output('shot-type-dd', 'value'),
    Input('player-dd', 'value'),
)
def update_shot_types(player_val):
    subset = meta[meta['player'] == player_val]
    types = sorted(subset['shot_type'].unique().tolist())
    options = [{'label': t, 'value': t} for t in types]
    value = types[0] if types else None
    return options, value


@app.callback(
    Output('shot-id-dd', 'options'),
    Output('shot-id-dd', 'value'),
    Input('player-dd', 'value'),
    Input('shot-type-dd', 'value'),
    Input('shot-id-dd', 'value'),
)
def update_shot_ids(player_val, shot_type_val, current_shot_id):
    subset = meta[(meta['player'] == player_val) & (meta['shot_type'] == shot_type_val)]
    ids = subset['shot_id'].astype(int).tolist()
    options = [{'label': str(i), 'value': int(i)} for i in ids]
    if not ids:
        return options, None

    if current_shot_id is not None and int(current_shot_id) in ids:
        value = int(current_shot_id)
    else:
        value = int(ids[0])
    return options, value


@app.callback(
    Output('shot-graph', 'figure'),
    Output('selection-readout', 'children'),
    Input('player-dd', 'value'),
    Input('shot-type-dd', 'value'),
    Input('shot-id-dd', 'value'),
)
def update_figure(player_val, shot_type_val, shot_id_val):
    subset = meta[(meta['player'] == player_val) & (meta['shot_type'] == shot_type_val)]
    if subset.empty:
        out = go.Figure()
        out.update_layout(title='No shots found for current player/shot type selection.')
        return out, 'No matching shots for current selection.'

    valid_ids = subset['shot_id'].astype(int).tolist()
    if shot_id_val is None or int(shot_id_val) not in valid_ids:
        sid = int(valid_ids[0])
    else:
        sid = int(shot_id_val)

    fig, err = _make_single_shot_panel(sid)
    if err:
        out = go.Figure()
        out.update_layout(title=str(err))
        return out, f'Selected: player={player_val}, shot_type={shot_type_val}, shot_id={sid} | ERROR: {err}'

    readout = f'Selected: player={player_val}, shot_type={shot_type_val}, shot_id={sid}'
    return fig, readout


def _pick_open_port(start=8060, end=8100):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    raise RuntimeError('No open port found in range 8060-8100.')


HOST = '0.0.0.0'
PORT = _pick_open_port()
print(f'Dash app starting on {HOST}:{PORT}')
print(f'Forward port {PORT} in VS Code Ports and open the forwarded URL.')
app.run(host=HOST, port=PORT, jupyter_mode='external', debug=False)